# Ising Model Jupyter Notebook

## This notebook is divided up into several sections, and the sections are designed to investigate the following:

1. Validate the 2D 2-spin ising model using parallelised simulation runs whilst changing the temperature to calculate the onsager critical temperature, and calculate the onsager critical exponents using the extracted simulation data, using both the Metropolis and Wolff Algorithms.
   
2. Use parallelised simulation runs again whilst changing the lattice size to calculate the value of $C_0$, the onsager critical amplitude.

3. Visualise the effects of changing the initial conditions of a generic 64x64 ising lattice using both the metropolis and wolff algorithms. Please note: The Wolff Algorithm was not designed to account for magnetic fields and hence the effects of changing the magnetic induction field are only investigated using the Metrolpolis 

# Main imports

In [1]:
### Imports ###

# Calculations 
import numpy as np

# Constants
from scipy.constants import mu_0

# Plotting and data fitting
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from scipy import stats

# File reading and config
import os
from tqdm.auto import tqdm

# Multiprocessing
from multiprocessing import Pool

# Sanity check: checking that all CPU cores are available for multiprocessing - Checks that os works properly

In [2]:
# Defining a global CPU core count
n_cores = os.cpu_count()

if type(n_cores) != int: # Must have an integer number of cores in any CPU
    
    raise ValueError('Critical error with python environment: os package has not registered all the CPU cores, please check os is working correctly.')

else:
    print('All CPU cores detected successfully.')
    print(f'Number of available CPU cores for multiprocessing is {n_cores}.')

All CPU cores detected successfully.
Number of available CPU cores for multiprocessing is 24.


# Sanity check - Check that the helper multiprocessing files are in place

In [3]:
# Checking that all the parallelisable worker files exist
if os.path.exists('parallelised_lattice_side_length_investigation_functions_MMC.py') != True or os.path.exists('parallelised_temperature_investigation_functions_MMC.py') != True or os.path.exists('parallelised_temperature_investigation_functions_WMC.py') != True or os.path.exists('parallelised_lattice_side_length_investigation_functions_WMC.py') != True:

    raise ValueError('Critical error with multiprocessing files: The files required for multiprocessing are either damaged or not in the same directory as this code.\nPlease ensure parallelised_lattice_side_length_investigation_functions_MMC.py and parallelised_temperature_investigation_functions_MMC.py are both in the same directory as this notebook before restarting the notebook kernel and re-running.')

else:
    print('All multiprocessing files successfully detected. Importing parallelisable functions and proceeding with runs.')

    # Importing the metropolis algorithm parallelisation functions
    from parallelised_temperature_investigation_functions_MMC import run_MMC_temp_investigation_worker
    from parallelised_lattice_side_length_investigation_functions_MMC import run_MMC_lattice_side_length_investigation_worker

    # Importing the wolff algorithm parallelisation functions
    from parallelised_lattice_side_length_investigation_functions_WMC import run_WMC_lattice_side_length_investigation_worker
    from parallelised_temperature_investigation_functions_WMC import run_WMC_temp_investigation_worker

All multiprocessing files successfully detected. Importing parallelisable functions and proceeding with runs.


# Function to read in parameter file for single ising model simulation (either wolff or metropolis)

In [4]:
def read_2D_ising_parameter_file(parameter_file):

    """Reads in the parameter file after checking it is there, and returns the parameters in the following formats:

    filename = string
    Lattice side length = int
    Magnetic field strength = float
    temperature = float
    Coupling constant J = float
    k_b (boltzmann constant) = float
    u_b (bohr magneton) = float
    number of steps = int
    
    This function is borrowed from the previous verlet integrator code, which also reads parameter files using the exact same method."""

    # Checking if the file is in the same directory as this notebook
    if os.path.exists(parameter_file) is False:

        raise ValueError('File is not in the same directory as this notebook. Please move the file to the same directory as the notebook and re-run.')

    # Empty dictionary to store the parameters
    parameter_storage_dict = {}

    with open(parameter_file, 'r') as f:

        for txtline in f:

            # Skipping empty lines and lines starting with a hashtag
            txtline = txtline.strip() # Removing spaces - Makes things easier
            if not txtline or txtline.startswith('#'):

                continue

            # Detecting assigned parameters in the file
            if '=' in txtline:
                key, parameter = txtline.split('=', 1) # Split the line based on the first occurrence of = character
                key = key.strip()
                value = parameter.strip()

                # Assigning the parameter
                parameter_storage_dict[key] = parameter

    # Sanity checks for dictionary
    for key, parameter in zip(parameter_storage_dict.keys(), parameter_storage_dict.values()):

        if key == None or parameter == None:
            raise ValueError('Parameter file is faulty: Conatains variables that point to no values or values with no assigned variable.')
            
        else:
            continue

    # Extracting parameters to return if the file contains no unassigned variables
    # Extracting filename
    filename = parameter_storage_dict.get('filename')
    filename = filename.strip() # Important to remove spaces - OneDrive has backup issues if the filename has spaces.

    # Extracting simulation numerical parameters
    lattice_side_length = int(parameter_storage_dict['side_length'])
    magnetic_field_B = float(parameter_storage_dict['magnetic_field_B'])
    temperature = float(parameter_storage_dict['temperature'])
    exchange_constant_J = float(parameter_storage_dict['exchange_constant_J'])
    k_b = float(parameter_storage_dict['k_b'])
    u_b = float(parameter_storage_dict['u_b'])
    nsteps = int(parameter_storage_dict['nsteps'])

    # Returning parameters in the same sequence as the run_integrator function takes them
    return lattice_side_length, magnetic_field_B, temperature, exchange_constant_J, k_b, u_b, nsteps, filename

# Function to read in parameter file for parallelised critical temperature investigation simulation (either wolff or metropolis)

In [5]:
def read_tcrit_investigative_file(parameter_file):

    """Reads in the parameter file after checking it is there, and returns the parameters in the following formats:

    Standard parameters:
    
    filename = string
    Lattice side length = int
    Magnetic field strength = float
    temperature = float
    Coupling constant J = float
    k_b (boltzmann constant) = float
    u_b (bohr magneton) = float
    number of steps per run = int

    Investigative parameters for tcrit:
    
    temperature step between runs = float
    number of runs = int
    
    This function is borrowed from the previous verlet integrator code, which also reads parameter files using the exact same method. """

    # Checking if the file is in the same directory as this notebook
    if os.path.exists(parameter_file) is False:

        raise ValueError('File is not in the same directory as this notebook. Please move the file to the same directory as the notebook and re-run.')

    # Empty dictionary to store the parameters
    parameter_storage_dict = {}

    with open(parameter_file, 'r') as f:

        for txtline in f:

            # Skipping empty lines and lines starting with a hashtag
            txtline = txtline.strip() # Removing spaces - Makes things easier
            if not txtline or txtline.startswith('#'):

                continue

            # Detecting assigned parameters in the file
            if '=' in txtline:
                key, parameter = txtline.split('=', 1) # Split the line based on the first occurrence of = character
                key = key.strip()
                value = parameter.strip()

                # Assigning the parameter
                parameter_storage_dict[key] = parameter

    # Sanity checks for dictionary
    for key, parameter in zip(parameter_storage_dict.keys(), parameter_storage_dict.values()):

        if key == None or parameter == None:
            raise ValueError('Parameter file is faulty: Conatains variables that point to no values or values with no assigned variable.')
            
        else:
            continue

    # Extracting parameters to return if the file contains no unassigned variables
    # Extracting filename
    filename = parameter_storage_dict.get('filename')
    filename = filename.strip() # Important to remove spaces - OneDrive has backup issues if the filename has spaces.

    # Extracting simulation numerical parameters
    lattice_side_length = int(parameter_storage_dict['side_length'])
    magnetic_field_B = float(parameter_storage_dict['magnetic_field_B'])
    initial_temperature = float(parameter_storage_dict['initial_temperature'])
    temp_step = float(parameter_storage_dict['temp_step'])
    exchange_constant_J = float(parameter_storage_dict['exchange_constant_J'])
    k_b = float(parameter_storage_dict['k_b'])
    u_b = float(parameter_storage_dict['u_b'])
    nsteps = int(parameter_storage_dict['nsteps'])
    nruns = int(parameter_storage_dict['nruns'])

    # Returning parameters in the same sequence as the run_integrator function takes them
    return lattice_side_length, magnetic_field_B, initial_temperature, temp_step, exchange_constant_J, k_b, u_b, nsteps, nruns, filename

# Function to read in parameter file for parallelised lattice side length change simulation (either wolff or metropolis)

In [6]:
def read_lattice_side_length_investigative_file(parameter_file):

    """Reads in the parameter file after checking it is there, and returns the parameters in the following formats:

    Standard parameters:
    
    filename = string
    Lattice side length = int
    Magnetic field strength = float
    temperature = float
    Coupling constant J = float
    k_b (boltzmann constant) = float
    u_b (bohr magneton) = float
    number of steps per run = int

    Investigative parameters for investigating the change of lattice size:
    
    side length step between runs = int
    number of runs = int
    
    This function is borrowed from the previous verlet integrator code, which also reads parameter files using the exact same method """

    # Checking if the file is in the same directory as this notebook
    if os.path.exists(parameter_file) is False:

        raise ValueError('File is not in the same directory as this notebook. Please move the file to the same directory as the notebook and re-run.')

    # Empty dictionary to store the parameters
    parameter_storage_dict = {}

    with open(parameter_file, 'r') as f:

        for txtline in f:

            # Skipping empty lines and lines starting with a hashtag
            txtline = txtline.strip() # Removing spaces - Makes things easier
            if not txtline or txtline.startswith('#'):

                continue

            # Detecting assigned parameters in the file
            if '=' in txtline:
                key, parameter = txtline.split('=', 1) # Split the line based on the first occurrence of = character
                key = key.strip()
                value = parameter.strip()

                # Assigning the parameter
                parameter_storage_dict[key] = parameter

    # Sanity checks for dictionary
    for key, parameter in zip(parameter_storage_dict.keys(), parameter_storage_dict.values()):

        if key == None or parameter == None:
            raise ValueError('Parameter file is faulty: Conatains variables that point to no values or values with no assigned variable.')
            
        else:
            continue

    # Extracting parameters to return if the file contains no unassigned variables
    # Extracting filename
    filename = parameter_storage_dict.get('filename')
    filename = filename.strip() # Important to remove spaces - OneDrive has backup issues if the filename has spaces.

    # Extracting simulation numerical parameters
    initial_lattice_side_length = int(parameter_storage_dict['initial_side_length'])
    magnetic_field_B = float(parameter_storage_dict['magnetic_field_B'])
    temperature = float(parameter_storage_dict['temperature'])
    lattice_side_length_step = int(parameter_storage_dict['lattice_side_length_step']) # Forcing lattice side length step to be an int
    exchange_constant_J = float(parameter_storage_dict['exchange_constant_J'])
    k_b = float(parameter_storage_dict['k_b'])
    u_b = float(parameter_storage_dict['u_b'])
    nsteps = int(parameter_storage_dict['nsteps'])
    nruns = int(parameter_storage_dict['nruns'])

    # Returning parameters in the same sequence as the run_integrator function takes them
    return initial_lattice_side_length, magnetic_field_B, temperature, lattice_side_length_step, exchange_constant_J, k_b, u_b, nsteps, nruns, filename

# Function to initialise a random 2D lattice. All the ising model simulations start from a random system

In [7]:
def make_random_2D_lattice(lattice_side_length, generator):
    
    """Produces a randomly generated 2D lattice of +1 or -1 spins using a specific random seed and numpy's random number generator."""
    
    # PCG64DXSM is the most random number generator available in numpy.
    # True randomness is achieved by passing no seed - Seed is therefore sourced from the operating system 
    
    random_2D_lattice = generator.choice([1, -1], size=(lattice_side_length, lattice_side_length))

    return random_2D_lattice

# Function to compute system energy change when a spin is flipped - Needed for metropolis algorithm to work. This function is only ever used for the metropolis algorithm function.

In [8]:
def calculate_energy_MMC(lattice_side_length, random_2D_lattice, magnetic_field_B, temperature, exchange_constant_J, u_b, x_coord, y_coord):

    """Function for Metropolis Monte Carlo Algorithm: 
    
    Uses a nearest neighbour approach to calculate the energy change when a random spin is flipped.
    
    DOES NOT SUM OVER THE ENTIRE LATTICE TO CALCULATE ENERGY - NEAREST NEIGHBOURS ONLY.
    
    Requires the lattice, magnetic field, temperature, exchange constant and two randomly generated coordinates, generated by the MMC function 
    (Metropolis Monte Carlo).
    
    Periodic boundary conditions are enforced to keep the model physically accurate."""

    ## Ising model hamiltonian for 1 spin and its nearest neighbours using periodic boundary conditions ##

    # Nearest neighbours term of the hamiltonian
    nearest_neighbours_term = (-exchange_constant_J) * random_2D_lattice[x_coord, y_coord] * np.sum([random_2D_lattice[x_coord, (y_coord-1)%lattice_side_length], random_2D_lattice[x_coord, (y_coord+1)%lattice_side_length], random_2D_lattice[(x_coord-1)%lattice_side_length, y_coord], random_2D_lattice[(x_coord+1)%lattice_side_length, y_coord]])

    # Magnetic term of the hamiltonian
    magnetic_term = u_b * random_2D_lattice[x_coord, y_coord] * magnetic_field_B 

    # The energy is given by the subtraction of the magnetic term from the nearest neighbours term
    spin_flip_energy = nearest_neighbours_term - magnetic_term

    return spin_flip_energy

# Function for performing a single metropolis step

In [9]:
def MMC_step(lattice_side_length, random_2D_lattice, magnetic_field_B, temperature, k_b, u_b, exchange_constant_J, generator):

    """Performs a monte-carlo step - Calculates energies before and after the spin change and compares a randomly generated number to 
    the maxwell boltzmann distribution"""

    # PCG64DXSM is the most random number generator available in numpy.
    # True randomness is achieved by passing no seed - Seed is therefore sourced from the operating system.

    # Set of random coordinates
    x_coord = generator.integers(0, lattice_side_length)
    y_coord = generator.integers(0, lattice_side_length)

    # Calculating initial_energy
    old_energy = calculate_energy_MMC(lattice_side_length, random_2D_lattice, magnetic_field_B, temperature, exchange_constant_J, u_b, x_coord, y_coord)

    # Selecting a spin and flipping it 
    random_2D_lattice[x_coord, y_coord] *= -1

    # Calculating new energy
    new_energy = calculate_energy_MMC(lattice_side_length, random_2D_lattice, magnetic_field_B, temperature, exchange_constant_J, u_b, x_coord, y_coord)
    
    # Calculating energy change
    energy_change = new_energy - old_energy

    if energy_change > 0 and np.exp(-energy_change / (k_b * temperature)) < generator.random():

        # Undoing the flip if it isnt probable enough to happen
        random_2D_lattice[x_coord, y_coord] *= -1

    else:
        pass # Keep the spin flipped if the condition is not met 

    return random_2D_lattice

# Function to compute a Wolff Monte Carlo Step

In [10]:
def WMC_step(lattice_side_length, random_2D_lattice, magnetic_field_B, temperature, k_b, u_b, exchange_constant_J, generator):

    """Performs a Wolff monte carlo step as seen on page 5 of G. T. Barkema and M. E. J. Newman 1997.

    Grows a cluster within that lattice, and then decides whether to flip an entire cluster or not rather than a singluar spin.
    
    Copy-pasted text from page 5:
    
    1) Choose a seed spin at random from the lattice.
    
    2) Look in turn at each of the neighbours of that spin. If they are pointing
    in the same direction as the seed spin, add them to the cluster with
    probability Padd = 1− e−2βJ. 
    
    3) For each spin that was added in the last step, examine each of its neigh-
    bours to find the ones which are pointing in the same direction and add
    each of them to the cluster with the same probability Padd. (Notice that
    as the cluster becomes larger we may find that some of the neighbours
    are already members of the cluster, in which case obviously you don’t
    have to consider adding them again. Also, some of the spins may have
    been considered for addition before, as neighbours of other spins in the
    cluster, but rejected. In this case, they get another chance to be added to
    the cluster on this step.) This step is repeated as many times as necessary
    until there are no spins left in the cluster whose neighbours have not been
    considered for inclusion in the cluster.
    
    4) Flip all the spins in the cluster."""
    
    ## Cluster-growth probability from equation 5 in the 1997 paper ##

    # Probability a spin is added to the cluster as it grows in the wolff algorithm
    # Note: Magnetic field isnt included in either of these terms because the standard wolff algorithm cant account for B-fields
    beta = 1/(k_b*temperature)
    p_add = 1 - np.exp(-2*beta*exchange_constant_J)

    ## Choosing a random spin, re-used code from the MMC_step ##

    # Set of random coordinates
    x_coord = generator.integers(0, lattice_side_length)
    y_coord = generator.integers(0, lattice_side_length)

    # Selecting the spin
    cluster_origin_spin = random_2D_lattice[x_coord, y_coord]

    ## Making the cluster ##

    # Cluster cannot contain duplicates, so a set is used (the set datatype CANNOT contain duplicates).
    cluster = set()

    # Adding the initial spin to the cluster
    cluster.add((x_coord, y_coord))

    # Making a list of spins that are to be compared in the cluster to see if new spins can be added to it.
    # Spins are checked later to see if they can be added based on the selection probability p_add
    cluster_neighbour_comparison_spins_list = [(x_coord, y_coord)]

    # Nearest neighbour boundary conditions
    nearest_neighbours_conditions = [(1, 0), (-1, 0), (0, 1), (0, -1)]

    ## Cluster growth loop ##

    # When this list empties itself at the end of the loop, the cluster is fully grown
    while cluster_neighbour_comparison_spins_list:

        # Removing the last element off the comparison list, so that its neighbours can be checked.
        initial_x_coord, initial_y_coord = cluster_neighbour_comparison_spins_list.pop()

        # Growing the cluster (adding spins to it based on the positions and spin orientation)
        for x_coord_add, y_coord_add in nearest_neighbours_conditions:

            # Coordinates to add to the cluster
            n_x_coord_to_add = (initial_x_coord + x_coord_add) % lattice_side_length
            n_y_coord_to_add = (initial_y_coord + y_coord_add) % lattice_side_length

            # Growing the cluster based on neighbours if the spins are:
            # 1: Neighbours are not already in the cluster
            # 2: Only has the same z axis orientation as the original  
            if (n_x_coord_to_add, n_y_coord_to_add) not in cluster:
                
                # Only adding spins to the cluster if they are pointing in the same direction (so mathematically equal to the seed spin).
                if random_2D_lattice[n_x_coord_to_add, n_y_coord_to_add] == cluster_origin_spin:

                    # Add to cluster with the previously calculated probability p_add
                    if generator.random() < p_add:

                        # Adding non-duplicate elements to the cluster
                        cluster.add((n_x_coord_to_add, n_y_coord_to_add))

                        # Adding to neighbour comparison list
                        cluster_neighbour_comparison_spins_list.append((n_x_coord_to_add, n_y_coord_to_add))

    ## Flipping the entire cluster ##
    
    for x_coord_to_flip, y_coord_to_flip in cluster:
        
        random_2D_lattice[x_coord_to_flip, y_coord_to_flip] *= -1

    return random_2D_lattice

# Function to calculate total lattice energy 

In [11]:
def calculate_lattice_energy_per_spin(lattice, lattice_side_length, exchange_constant_J, magnetic_field_B, u_b):

    """Calculates the total energy of the entire lattice using the same formula as the energy used to calculate a spin flip, just
    applied to the entire lattice than a specific set of spin coordinate"""

    # Uses np.roll to get periodic summation - Vectorised method of calculating the energy which is much faster than nested for loops
    periodic_boundary_sum = np.roll(lattice, 1, axis=0) + np.roll(lattice, -1, axis=0) + np.roll(lattice, 1, axis=1) + np.roll(lattice, -1, axis=1)

    # Nearest neighbour term in the hamiltonian
    nearest_neighbour_term = (-exchange_constant_J/2)*np.sum(lattice*periodic_boundary_sum)

    # Magnetic term in the hamiltonian
    magnetic_term = u_b*np.sum(magnetic_field_B*lattice)

    # Total lattice energy is given by the subtraction of the magnetic term from the hamiltonian term
    total_lattice_energy = nearest_neighbour_term - magnetic_term

    # Calculating the energy per spin
    energy_per_spin = total_lattice_energy / (lattice_side_length**2)

    return energy_per_spin

# Function to calculate total lattice magnetisation

In [12]:
def calculate_magnetisation_per_spin(lattice, lattice_side_length):

    """Calculates the lattice magnetisation for a single 2D lattice."""

    # Using formula from the instructions to calculate the lattice magnetisation
    lattice_magnetisation_per_spin = (1/lattice_side_length**2)*np.sum(lattice)
    
    return lattice_magnetisation_per_spin

# Function to calculate total lattice internal magnetic field

In [13]:
def calculate_internal_magnetic_field_per_spin(lattice, lattice_side_length, magnetic_field_B):

    """Calculates the internal magentic field strength of a single 2D lattice. Calls the lattice magnetisation function
    in order to get the lattice magnetisation."""

    # Extracting the magnetisation using the function to calculate it
    lattice_magnetisation_per_spin = calculate_magnetisation_per_spin(lattice, lattice_side_length)

    # Calculating the internal magnetic field using the magnetisation, bohr magneton and lattice magnetisation
    lattice_internal_magnetic_field_per_spin = (1/mu_0)*(magnetic_field_B-lattice_magnetisation_per_spin)

    return lattice_internal_magnetic_field_per_spin

# Function to calculate linear regression fit - Needed for straight line diagnostic plots

In [14]:
def calculate_linear_regress_gradient_fit(xs, ys):

    """Calculates the linear regression fit using scipy's lineregress fit library, and returns gradient, intercept and the error on the fit."""

    # Sanity check on the array sizes 
    if xs.size != ys.size:

        raise ValueError('Arrays are not of the same size, please check the code that makes the arrays that were inputted into this function.')

    else:
    
        # Running the two numpy arrays through the linear regression library 
        result = stats.linregress(xs, ys)
    
        ## Extracting results to be returned ##
    
        # Slope and its error
        slope = result.slope
        slope_error = result.stderr
    
        # Intercept and its error
        intercept = result.intercept
        intercept_error = result.intercept_stderr

    return slope, slope_error, intercept, intercept_error

# Function to calculate a straight line fit - Needed for plotting lines of best fit

In [15]:
def straight_line(x, m, c):

    """Function used to calculate the straight line fit"""

    return (m*x)+c    

# Function to run a single metropolis algorithm simulation with end system visualisation

In [16]:
def run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file, **kwargs):

    """Uses the reading function to read in the parameters, 
    and then runs the ising model and returns the final lattice configuration along with any plots.
    
    Stores the lattices in a list of arrays so they can be accessed and analysed later."""

    # Function random generator
    generator = np.random.Generator(np.random.PCG64DXSM())

    # Condition to show / hide the diagnostic plots
    plot_mode = kwargs.get('plot_mode', None)

    # Setting up ICS and the lattice from the file 

    # Extracting ICS
    lattice_side_length, magnetic_field_B, temperature, exchange_constant_J, k_b, u_b, nsteps, filename = read_2D_ising_parameter_file(parameter_file)

    # Initialising a random 2D lattice
    initial_lattice = make_random_2D_lattice(lattice_side_length, generator)

    # Creating the lattice used for the ising model simulation itself, leaving the initial lattice separate and unchanged.
    lattice = initial_lattice.copy()

    # Printing parameters as a sanity check - Make sure the file is read in correctly.
    print(f'Running single Metropolis Algorithm simulation with parameters:')
    print(f'temperature={temperature}K, lattice_side_length={lattice_side_length}, magnetic_field={magnetic_field_B}, exchange_constant_J={exchange_constant_J}, k_b={k_b}, u_b={u_b}, nsteps={nsteps}\n')

    # Burn-in limit for calculations of variances - This is to account for equilibration
    burnin_step_proportion = 0.6
    burnin_equilibration_steps_threshold = int(burnin_step_proportion*nsteps)

    # Lists to store end parameters generated during the main simulation loop
    lattice_energy_per_spin_list = []
    lattice_magnetisation_per_spin_list = []
    lattice_internal_magnetic_field_per_spin_list = []
    lattice_magnetic_susceptibility_per_spin_list = []
    lattice_heat_capacity_per_spin_list = []

    # Number of equilibrated samples
    n_equilibrated = 0

    # Initialising starting variances for the welford algorithm implementation in the main simulation loop
    running_mean_magnetisation_per_spin = 0.0
    running_mean_magnetisation_per_spin_2 = 0.0
    running_mean_energy_per_spin = 0.0
    running_mean_energy_per_spin_2 = 0.0

    # Number of equilibrated samples
    n_equilibrated = 0
    
    ### MAIN SIMULATION LOOP ###
    for i in tqdm(range(nsteps), desc='Running MMC Ising Model:'):
        
        # Reassigning the lattice with every step, making the ising model run. 
        lattice = MMC_step(lattice_side_length, lattice, magnetic_field_B, temperature, k_b, u_b, exchange_constant_J, generator)

        # Calling functions on the lattice every timestep to calculate the quantities of interest
        lattice_energy_per_spin = calculate_lattice_energy_per_spin(lattice, lattice_side_length, exchange_constant_J, magnetic_field_B, u_b)
        lattice_magnetisation_per_spin = calculate_magnetisation_per_spin(lattice, lattice_side_length) # In the metropolis algorithm there is no risk of the entire lattice turning into a cluster being flipped so the true magnetisation (not the absolute value) can be used here
        lattice_internal_magnetic_field_per_spin = calculate_internal_magnetic_field_per_spin(lattice, lattice_side_length, magnetic_field_B)

        # Appending values to lists
        lattice_energy_per_spin_list.append(lattice_energy_per_spin)
        lattice_magnetisation_per_spin_list.append(lattice_magnetisation_per_spin)
        lattice_internal_magnetic_field_per_spin_list.append(lattice_internal_magnetic_field_per_spin)
        
        # Only begin the welford loop after equilibration
        if i > burnin_equilibration_steps_threshold:

            # Computing number of equilibrated samples
            n_equilibrated += 1

            ## WELFORD ALGORITHM IMPLEMENTATION - ONLY RUNS AFTER EQUILIBRATION HAS BEEN ACHIEVED ##
        
            # Two staged approach to extracting variances - Magnetisation
                
            # Stage 1
            magnetisation_diff = lattice_magnetisation_per_spin - running_mean_magnetisation_per_spin
            running_mean_magnetisation_per_spin += magnetisation_diff / n_equilibrated
                
            # Stage 2
            magnetisation_diff_2 = lattice_magnetisation_per_spin - running_mean_magnetisation_per_spin
            running_mean_magnetisation_per_spin_2 += magnetisation_diff * magnetisation_diff_2
        
            # Two staged approach to extracting variances - Energy
        
            # Stage 1
            energy_diff = lattice_energy_per_spin - running_mean_energy_per_spin
            running_mean_energy_per_spin += energy_diff / n_equilibrated
                
            # Stage 2
            energy_diff_2 = lattice_energy_per_spin - running_mean_energy_per_spin
            running_mean_energy_per_spin_2 += energy_diff * energy_diff_2
        
            if n_equilibrated > 1:
        
                # Calculating the variances for magnetisation and energy
                magnetisation_per_spin_running_variance = running_mean_magnetisation_per_spin_2 / n_equilibrated
                energy_per_spin_running_variance = running_mean_energy_per_spin_2 / n_equilibrated
        
                # Appending magnetic susceptibility and heat capacity
                lattice_magnetic_susceptibility_per_spin_list.append((1/(k_b*temperature)) * magnetisation_per_spin_running_variance)
                lattice_heat_capacity_per_spin_list.append(((1/(k_b*temperature**2)) * energy_per_spin_running_variance))

    # Converting to arrays - useful for calculations later
    lattice_energy_per_spin_array = np.asarray(lattice_energy_per_spin_list)
    lattice_magnetisation_per_spin_array = np.asarray(lattice_magnetisation_per_spin_list)
    lattice_internal_magnetic_field_per_spin_array = np.asarray(lattice_internal_magnetic_field_per_spin_list)
    lattice_magnetic_susceptibility_per_spin_array = np.asarray(lattice_magnetic_susceptibility_per_spin_list)
    lattice_heat_capacity_per_spin_array = np.asarray(lattice_heat_capacity_per_spin_list)
        
    ## Making main lattice figure ##
    
    fig, ax = plt.subplots(figsize=(4,4), dpi=300)

    # cmap
    blue_yellow_cmap = colors.ListedColormap(['blue', 'yellow'])

    # Plotting
    ising_2D_plot = ax.imshow(lattice, cmap=blue_yellow_cmap)

    # Formatting
    ax.set_title('2D Ising Model Lattice Plot, MMC')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
        
    cbar = fig.colorbar(ising_2D_plot, ax=ax, ticks=[-1, 1], shrink=0.75)
    cbar.ax.set_yticklabels(['Spin = -1', 'Spin = 1'])

    # Saving figure 
    fig.savefig(f'End lattice configuration, filename={filename}, MMC.png', dpi=300)

    ## Making figures for the quantities of interest against the number of timesteps ##

    # Energy per spin vs timestep
    fig1, ax1 = plt.subplots(figsize=(6,4), dpi=300)
    ax1.plot(range(0, nsteps), lattice_energy_per_spin_array)
    ax1.set_title('Lattice energy per spin vs timestep, MMC')
    ax1.set_xlabel('N Monte-Carlo steps')
    ax1.set_ylabel('Energy per spin / J')
    ax1.grid(ls=':', color='0.9')

    # Saving figure
    fig1.savefig(f'Lattice energy per spin vs timestep, filename={filename}, MMC.png', dpi=300)

    # Magnetisation per spin vs timestep
    fig2, ax2 = plt.subplots(figsize=(6,4), dpi=300)
    ax2.plot(range(0, nsteps), lattice_magnetisation_per_spin_array)
    ax2.set_title('Lattice magnetisation per spin vs timestep, MMC')
    ax2.set_xlabel('N Monte-Carlo steps')
    ax2.set_ylabel('Magnetisation per spin')
    ax2.grid(ls=':', color='0.9')

    # Saving figure
    fig2.savefig(f'Lattice magnetisation per spin vs timestep, filename={filename}, MMC.png', dpi=300)
    
    # Lattice internal magnetic field per spin vs timestep
    fig3, ax3 = plt.subplots(figsize=(6,4), dpi=300)
    ax3.plot(range(0, nsteps), lattice_internal_magnetic_field_per_spin_array)
    ax3.set_title('Lattice internal magnetic field per spin vs timestep, MMC')
    ax3.set_xlabel('N Monte-Carlo steps')
    ax3.set_ylabel('Internal magnetic field per spin')
    ax3.grid(ls=':', color='0.9')

    # Saving figure
    fig3.savefig(f'Internal magnetic field per spin vs timestep, filename={filename}, MMC.png', dpi=300)

    ## Variables that require variances ##

    # Magnetic susceptibility per spin vs timestep
    fig4, ax4 = plt.subplots(figsize=(6,4), dpi=300)
    ax4.plot(range(burnin_equilibration_steps_threshold+2, nsteps), lattice_magnetic_susceptibility_per_spin_array) # Skipping the first step, since there is no variance between the first lattice and itself.
    ax4.set_title('Lattice magnetic susceptibility per spin vs timestep, MMC')
    ax4.set_xlabel('N Monte-Carlo steps')
    ax4.set_ylabel('Magnetic susceptibility per spin')
    ax4.grid(ls=':', color='0.9')

    # Saving figure
    fig4.savefig(f'Magnetic susceptibility per spin vs timestep, filename={filename}, MMC.png', dpi=300)

    # Heat Capacity per spin vs timestep
    fig5, ax5 = plt.subplots(figsize=(6,4), dpi=300)
    ax5.plot(range(burnin_equilibration_steps_threshold+2, nsteps), lattice_heat_capacity_per_spin_array) # Skipping the first step, since there is no variance between the first lattice and itself.
    ax5.set_title('Lattice heat capacity per spin vs timestep, MMC')
    ax5.set_xlabel('N Monte-Carlo steps')
    ax5.set_ylabel('Heat capacity per spin')
    ax5.grid(ls=':', color='0.9')

    # Saving figure
    fig5.savefig(f'Lattice heat capacity per spin vs timestep, filename={filename}, MMC.png', dpi=300)

    # This will be used on the validation plots to conserve memory
    if plot_mode == 'close':
        plt.close('all')
            
    # If no mode is defined, force the plots to display
    elif plot_mode == None:
        plt.show('all')

    # Return ising pixel plot and the end magnetisation, which is critical for the temperature checks.  
    return fig, ax, fig1, ax1, fig2, ax2, fig3, ax3, fig4, ax4, fig5, ax5

# Function to run a single wolff algorithm simulation with end system visualisation

In [17]:
def run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file, **kwargs):

    """Uses the reading function to read in the parameters, 
    and then runs the ising model and returns the final lattice configuration along with any plots.
    
    Stores the lattices in a list of arrays so they can be accessed and analysed later.
    
    Exactly the same as run_single_ising_model_simulation_with_lattice_visualisation_MMC, but uses the 
    Wolff algorithm instead of the met."""

    # Function random generator
    generator = np.random.Generator(np.random.PCG64DXSM())

    # Condition to show / hide the diagnostic plots
    plot_mode = kwargs.get('plot_mode', None)

    # Setting up ICS and the lattice from the file 

    # Extracting ICS
    lattice_side_length, magnetic_field_B, temperature, exchange_constant_J, k_b, u_b, nsteps, filename = read_2D_ising_parameter_file(parameter_file)

    # Initialising a random 2D lattice
    initial_lattice = make_random_2D_lattice(lattice_side_length, generator)

    # Creating the lattice used for the ising model simulation itself, leaving the initial lattice separate and unchanged.
    lattice = initial_lattice.copy()

    # Printing parameters as a sanity check - Make sure the file is read in correctly.
    print(f'Running single Wolff Algorithm simulation with parameters:')
    print(f'temperature={temperature}K, lattice_side_length={lattice_side_length}, magnetic_field={magnetic_field_B}, exchange_constant_J={exchange_constant_J}, k_b={k_b}, u_b={u_b}, nsteps={nsteps}\n')

    # Burn-in limit for calculations of variances - This is to account for equilibration
    burnin_step_proportion = 0.6
    burnin_equilibration_steps_threshold = int(burnin_step_proportion*nsteps)

    # Lists to store end parameters generated during the main simulation loop
    lattice_energy_per_spin_list = []
    lattice_magnetisation_per_spin_list = []
    lattice_internal_magnetic_field_per_spin_list = []
    lattice_magnetic_susceptibility_per_spin_list = []
    lattice_heat_capacity_per_spin_list = []

    # Number of equilibrated samples
    n_equilibrated = 0

    # Initialising starting variances for the welford algorithm implementation in the main simulation loop
    running_mean_magnetisation_per_spin = 0.0
    running_mean_magnetisation_per_spin_2 = 0.0
    running_mean_energy_per_spin = 0.0
    running_mean_energy_per_spin_2 = 0.0

    # Number of equilibrated samples
    n_equilibrated = 0
    
    ### MAIN SIMULATION LOOP ###
    for i in tqdm(range(nsteps), desc='Running WMC Ising Model:'):
        
        # Reassigning the lattice with every step, making the ising model run. 
        lattice = WMC_step(lattice_side_length, lattice, magnetic_field_B, temperature, k_b, u_b, exchange_constant_J, generator)

        ## CHANGE FROM run_single_ising_model_simulation_with_lattice_visualisation_MMC ##

        # All magnetic quantities (internal field H and magnetisation M) are absolute quantities instead of signed.
        # This is because after it equilibrates, the wolff algorithm will grow and flip a cluster
        # as big as the entire lattice, resulting in the plots flaring out and becoming unusable. 

        # Calling functions on the lattice every timestep to calculate the quantities of interest
        lattice_energy_per_spin = calculate_lattice_energy_per_spin(lattice, lattice_side_length, exchange_constant_J, magnetic_field_B, u_b)
        lattice_magnetisation_per_spin = np.abs(calculate_magnetisation_per_spin(lattice, lattice_side_length)) # Absolute magnetisation instead of standard magnetisation for wolff algorithm to guard against the effects of the entire lattice being turned into a cluster and being flipped
        lattice_internal_magnetic_field_per_spin = np.abs(calculate_internal_magnetic_field_per_spin(lattice, lattice_side_length, magnetic_field_B)) # Absolute magnetic field instead of standard magnetic field for wolff algorithm to guard against the effects of the entire lattice being turned into a cluster and being flipped

        # Appending values to lists
        lattice_energy_per_spin_list.append(lattice_energy_per_spin)
        lattice_magnetisation_per_spin_list.append(lattice_magnetisation_per_spin)
        lattice_internal_magnetic_field_per_spin_list.append(lattice_internal_magnetic_field_per_spin)
        
        # Only begin the welford loop after equilibration
        if i > burnin_equilibration_steps_threshold:

            # Computing number of equilibrated samples
            n_equilibrated += 1

            ## WELFORD ALGORITHM IMPLEMENTATION - ONLY RUNS AFTER EQUILIBRATION HAS BEEN ACHIEVED ##
        
            # Two staged approach to extracting variances - Magnetisation
                
            # Stage 1
            magnetisation_diff = lattice_magnetisation_per_spin - running_mean_magnetisation_per_spin
            running_mean_magnetisation_per_spin += magnetisation_diff / n_equilibrated
                
            # Stage 2
            magnetisation_diff_2 = lattice_magnetisation_per_spin - running_mean_magnetisation_per_spin
            running_mean_magnetisation_per_spin_2 += magnetisation_diff * magnetisation_diff_2
        
            # Two staged approach to extracting variances - Energy
        
            # Stage 1
            energy_diff = lattice_energy_per_spin - running_mean_energy_per_spin
            running_mean_energy_per_spin += energy_diff / n_equilibrated
                
            # Stage 2
            energy_diff_2 = lattice_energy_per_spin - running_mean_energy_per_spin
            running_mean_energy_per_spin_2 += energy_diff * energy_diff_2
        
            if n_equilibrated > 1:
        
                # Calculating the variances for magnetisation and energy
                magnetisation_per_spin_running_variance = running_mean_magnetisation_per_spin_2 / n_equilibrated
                energy_per_spin_running_variance = running_mean_energy_per_spin_2 / n_equilibrated
        
                # Appending magnetic susceptibility and heat capacity
                lattice_magnetic_susceptibility_per_spin_list.append((1/(k_b*temperature)) * magnetisation_per_spin_running_variance)
                lattice_heat_capacity_per_spin_list.append(((1/(k_b*temperature**2)) * energy_per_spin_running_variance))

    # Converting to arrays - useful for calculations later
    lattice_energy_per_spin_array = np.asarray(lattice_energy_per_spin_list)
    lattice_magnetisation_per_spin_array = np.asarray(lattice_magnetisation_per_spin_list)
    lattice_internal_magnetic_field_per_spin_array = np.asarray(lattice_internal_magnetic_field_per_spin_list)
    lattice_magnetic_susceptibility_per_spin_array = np.asarray(lattice_magnetic_susceptibility_per_spin_list)
    lattice_heat_capacity_per_spin_array = np.asarray(lattice_heat_capacity_per_spin_list)
        
    ## Making main lattice figure ##
    
    fig, ax = plt.subplots(figsize=(4,4), dpi=300)

    # cmap
    blue_yellow_cmap = colors.ListedColormap(['blue', 'yellow'])

    # Plotting
    ising_2D_plot = ax.imshow(lattice, cmap=blue_yellow_cmap)

    # Formatting
    ax.set_title('2D Ising Model Lattice Plot, WMC')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
        
    cbar = fig.colorbar(ising_2D_plot, ax=ax, ticks=[-1, 1], shrink=0.75)
    cbar.ax.set_yticklabels(['Spin = -1', 'Spin = 1'])

    # Saving figure 
    fig.savefig(f'End lattice configuration, filename={filename}, WMC.png', dpi=300)

    ## Making figures for the quantities of interest against the number of timesteps ##

    # Energy per spin vs timestep
    fig1, ax1 = plt.subplots(figsize=(6,4), dpi=300)
    ax1.plot(range(0, nsteps), lattice_energy_per_spin_array)
    ax1.set_title('Lattice energy per spin vs timestep, WMC')
    ax1.set_xlabel('N Monte-Carlo steps')
    ax1.set_ylabel('Energy per spin / J')
    ax1.grid(ls=':', color='0.9')

    # Saving figure
    fig1.savefig(f'Lattice energy per spin vs timestep, filename={filename}, WMC.png', dpi=300)

    # Magnetisation per spin vs timestep
    fig2, ax2 = plt.subplots(figsize=(6,4), dpi=300)
    ax2.plot(range(0, nsteps), lattice_magnetisation_per_spin_array)
    ax2.set_title('Lattice absolute magnetisation per spin vs timestep, WMC')
    ax2.set_xlabel('N Monte-Carlo steps')
    ax2.set_ylabel('Absolute magnetisation per spin')
    ax2.grid(ls=':', color='0.9')

    # Saving figure
    fig2.savefig(f'Lattice magnetisation per spin vs timestep, filename={filename}, WMC.png', dpi=300)
    
    # Lattice internal magnetic field per spin vs timestep
    fig3, ax3 = plt.subplots(figsize=(6,4), dpi=300)
    ax3.plot(range(0, nsteps), lattice_internal_magnetic_field_per_spin_array)
    ax3.set_title('Lattice absolute internal magnetic field per spin vs timestep, WMC')
    ax3.set_xlabel('N Monte-Carlo steps')
    ax3.set_ylabel('Absolute internal magnetic field per spin')
    ax3.grid(ls=':', color='0.9')

    # Saving figure
    fig3.savefig(f'Internal magnetic field per spin vs timestep, filename={filename}, WMC.png', dpi=300)

    ## Variables that require variances ##

    # Magnetic susceptibility per spin vs timestep
    fig4, ax4 = plt.subplots(figsize=(6,4), dpi=300)
    ax4.plot(range(burnin_equilibration_steps_threshold+2, nsteps), lattice_magnetic_susceptibility_per_spin_array) # Skipping the first step, since there is no variance between the first lattice and itself.
    ax4.set_title('Lattice magnetic susceptibility per spin vs timestep, WMC')
    ax4.set_xlabel('N Monte-Carlo steps')
    ax4.set_ylabel('Magnetic susceptibility per spin')
    ax4.grid(ls=':', color='0.9')

    # Saving figure
    fig4.savefig(f'Magnetic susceptibility per spin vs timestep, filename={filename}, WMC.png', dpi=300)

    # Heat Capacity per spin vs timestep
    fig5, ax5 = plt.subplots(figsize=(6,4), dpi=300)
    ax5.plot(range(burnin_equilibration_steps_threshold+2, nsteps), lattice_heat_capacity_per_spin_array) # Skipping the first step, since there is no variance between the first lattice and itself.
    ax5.set_title('Lattice heat capacity per spin vs timestep, WMC')
    ax5.set_xlabel('N Monte-Carlo steps')
    ax5.set_ylabel('Heat capacity per spin')
    ax5.grid(ls=':', color='0.9')

    # Saving figure
    fig5.savefig(f'Lattice heat capacity per spin vs timestep, filename={filename}, WMC.png', dpi=300)

    # This will be used on the validation plots to conserve memory
    if plot_mode == 'close':
        plt.close('all')
            
    # If no mode is defined, force the plots to display
    elif plot_mode == None:
        plt.show('all')

    # Return ising pixel plot and the end magnetisation, which is critical for the temperature checks.  
    return fig, ax, fig1, ax1, fig2, ax2, fig3, ax3, fig4, ax4, fig5, ax5

# Function for running parallelised critical temperature and exponents investigation using the metropolis algorithm

In [18]:
def run_parallelised_temp_change_investigation_simulations_MMC(parameter_file, **kwargs):

    """Calls the run_single_temp_investigation_simulation(args) function N cores times per instance and packages the results up into lists.
    Once enough instances have been called that the full temperature sweep has been completed, the lists of mean magnetisations, 
    mean magnetic susceptibilities and mean heat capacities are converted into arrays and used for critical temperature and critical 
    exponent calculations.
    
    In order to run the run_single_temp_investigation_simulation(args) function in parallel, it uses all available cpu cores and the pool module 
    from multiprocessing package.
    
    Returns plots of magnetisations and magnetic susceptibility vs temperatures."""

    # Getting the plot mode - condition to show / hide the diagnostic plots
    plot_mode = kwargs.get('plot_mode', None)

    # Setting up ICS and the lattice from the file 
    lattice_side_length, magnetic_field_B, initial_temperature, temp_step, exchange_constant_J, k_b, u_b, nsteps, nruns, filename = read_tcrit_investigative_file(parameter_file)

    # Making a list to store temperatures
    run_temperatures_list = []

    # Appending to the list using temp step and the amount of runs
    for i in range(nruns):

        run_temperatures_list.append(initial_temperature + i*temp_step)

    # Turning the list into an array as this will be useful later
    run_temperatures_array = np.asarray(run_temperatures_list)

    ## Setting up parallelised simulation runs ##

    # Runs for worker function defined in the file 
    single_temp_run_args = [(temperature, lattice_side_length, magnetic_field_B, exchange_constant_J, k_b, u_b, nsteps) for temperature in run_temperatures_array]

    # Printing out initialisation message for parallelised simulations
    print(f'Parallelising {nruns} temperature investigation runs across {n_cores} CPU cores using the metropolis algorithm.')
    print(f'Temperature range: {run_temperatures_array[0]} --> {run_temperatures_array[-1]}.')
    print(f'Per-simulation parameters: temp_step={temp_step}K, lattice_side_length={lattice_side_length}, magnetic_field_B={magnetic_field_B}, exchange_constant_J={exchange_constant_J}, k_b={k_b}, u_b={u_b}, nsteps={nsteps}.')

    # Running multiple simulations in parallel
    with Pool(processes=os.cpu_count()) as pool:
        
        # Calling the imported parallelisable worker function to run multiple simulations at the same time using multiprocessing
        # This completes the runs across multiple temperatures much faster than using one core only
        per_run_mean_quantites = list(tqdm(pool.imap(run_MMC_temp_investigation_worker, single_temp_run_args), total=nruns, desc='Running parallelised metropolis algorithm temperature sweep:'))

    # Unpacking results from the list generated using multiprocessing
    # This has to be done in the same order as the run_single_temp_investigation_simulation(args) returns them in.
    per_run_mean_equilibrated_magnetisations_per_spin_array = np.array([mean_results[0] for mean_results in per_run_mean_quantites])
    per_run_mean_equilibrated_magnetic_susceptibilities_per_spin_array = np.array([mean_results[1] for mean_results in per_run_mean_quantites])
    per_run_mean_equilibrated_heat_capacities_per_spin_array = np.array([mean_results[2] for mean_results in per_run_mean_quantites])
    
    ## Critical temperature extraction ##
    
    # Method 1: Steepest drop in absolute magnetisation - Midpoint of the two temperatures between where this happens should yield phase transition temperature.
    consecutive_magnetisation_diffs = np.diff(per_run_mean_equilibrated_magnetisations_per_spin_array)
    largest_magnetisation_drop_index = np.argmin(consecutive_magnetisation_diffs)
    
    # Average of 2 values before and after the index for the most negative drop in magnetisation should yield the critical temperature
    tcrit_from_magnetisation = (run_temperatures_array[largest_magnetisation_drop_index] + run_temperatures_array[largest_magnetisation_drop_index+1])/2
    
    # Method 2: Index of the temperature where magnetic susceptibility peaks
    largest_magnetic_susceptibility_index = np.argmax(per_run_mean_equilibrated_magnetic_susceptibilities_per_spin_array)
    tcrit_from_magnetic_susceptibility = run_temperatures_array[largest_magnetic_susceptibility_index]

    # Method 3: Index of temperature where heat capacity peaks
    largest_heat_capacity_index = np.argmax(per_run_mean_equilibrated_heat_capacities_per_spin_array)
    tcrit_from_heat_capacity = run_temperatures_array[largest_heat_capacity_index]

    # Method 3: Average of methods 1, 2 and 3 - Most statistical accuracy
    average_tcrit = (tcrit_from_magnetisation + tcrit_from_magnetic_susceptibility + tcrit_from_heat_capacity)/3

    ## Storage of Onsager's calculated critical temperature and critical exponents for comparison and error calculations ##

    # Onsager 1944 critical temperature
    tcrit_onsager = 2.269185

    # Storing Onsager 1944 critical exponents 

    # Alpha critical exponent
    negative_onsager_critical_exponent = -0.4945

    # Beta critical exponent
    onsager_beta = 1/8

    # Gamma critical exponent
    onsager_gamma = -7/4

    ## Temperature regime filters ##

    # Above tcrit filter with 1% safety threshold.
    above_tcrit_filter = run_temperatures_array > average_tcrit*1.001

    # Below tcrit filter with 1% safety threshold.
    below_tcrit_filter = run_temperatures_array < average_tcrit*0.99

    ## Using reduced temperatures because they are dimensionless and thus ideal for power law scaling ##
    
    # Reduced temperatures using data above tcrit
    absolute_reduced_temperatures_above_tcrit = (run_temperatures_array[above_tcrit_filter] - average_tcrit) / average_tcrit

    # Reduced temperatures using data below tcrit
    absolute_reduced_temperatures_below_tcrit = (run_temperatures_array[below_tcrit_filter] - average_tcrit) / average_tcrit

    ## Heat capacity, magnetisation and susceptibility filtering ##

    # Separate heat capacities above and below tcrit
    heat_capacities_above_tcrit = per_run_mean_equilibrated_heat_capacities_per_spin_array[above_tcrit_filter]
    heat_capacities_below_tcrit = per_run_mean_equilibrated_heat_capacities_per_spin_array[below_tcrit_filter]

    # Separate magnetisations below tcrit. Only applies below tcrit
    magnetisations_below_tcrit = per_run_mean_equilibrated_magnetisations_per_spin_array[below_tcrit_filter]

    # Separate magnetic susceptibilities above and below tcrit
    magnetic_susceptibilities_above_tcrit = per_run_mean_equilibrated_magnetic_susceptibilities_per_spin_array[above_tcrit_filter]
    magnetic_susceptibilities_below_tcrit = per_run_mean_equilibrated_magnetic_susceptibilities_per_spin_array[below_tcrit_filter]

    ## Computing natural logarithms of absolute reduced temperatures above and below tcrit ##

    # np.abs is used to prevent the logarithm of a negative quantity from being computed. 
    ln_absolute_reduced_temperatures_above_tcrit = np.log(np.abs(absolute_reduced_temperatures_above_tcrit))
    ln_absolute_reduced_temperatures_below_tcrit = np.log(np.abs(absolute_reduced_temperatures_below_tcrit))

    ## Computing natural logarithm of magnetisation below tcrit

    # np.abs is used to prevent the logarithm of a negative quantity from being computed.
    ln_magnetisations_below_tcrit = np.log(np.abs(magnetisations_below_tcrit))

    ## Computing natural logarithms of magnetic susceptibilities above and below tcrit ##

    # np.abs is used to prevent the logarithm of a negative quantity from being computed.
    ln_magnetic_susceptibilities_above_tcrit = np.log(np.abs(magnetic_susceptibilities_above_tcrit))
    ln_magnetic_susceptibilities_below_tcrit = np.log(np.abs(magnetic_susceptibilities_below_tcrit))

    ## Computing the 5 different gradients ##

    # Gradient of ln heat capacity vs ln reduced temperatures above and below tcrit
    gradient_heat_capacity_above_tcrit, error_gradient_heat_capacity_above_tcrit, heat_capacity_intercept_above_tcrit, error_heat_capacity_intercept_above_tcrit = calculate_linear_regress_gradient_fit(ln_absolute_reduced_temperatures_above_tcrit, heat_capacities_above_tcrit)
    gradient_heat_capacity_below_tcrit, error_gradient_heat_capacity_below_tcrit, heat_capacity_intercept_below_tcrit, error_heat_capacity_intercept_below_tcrit = calculate_linear_regress_gradient_fit(ln_absolute_reduced_temperatures_below_tcrit, heat_capacities_below_tcrit)

    # Gradient of ln magnetisation vs ln reduced temperatures above and below tcrit
    gradient_magnetisation_below_tcrit, error_gradient_heat_capacity_below_tcrit, magnetisation_intercept_below_tcrit, error_magnetisation_intercept_below_tcrit = calculate_linear_regress_gradient_fit(ln_absolute_reduced_temperatures_below_tcrit, ln_magnetisations_below_tcrit)

    # Gradient of ln magnetic susceptibility vs ln reduced temperatures above and below tcrit
    gradient_magnetic_susceptibility_above_tcrit, error_gradient_magnetic_susceptibility_above_tcrit, magnetic_susceptibility_intercept_above_tcrit, error_magnetic_susceptibility_intercept_above_tcrit = calculate_linear_regress_gradient_fit(ln_absolute_reduced_temperatures_above_tcrit, ln_magnetic_susceptibilities_above_tcrit)
    gradient_magnetic_susceptibility_below_tcrit, error_gradient_magnetic_susceptibility_below_tcrit, magnetic_susceptibility_intercept_below_tcrit, error_magnetic_susceptibility_intercept_below_tcrit = calculate_linear_regress_gradient_fit(ln_absolute_reduced_temperatures_below_tcrit, ln_magnetic_susceptibilities_below_tcrit)

    ### Making plots ###

    # Generating array of x coordinates for plotting straight line for above tcrit
    x_temperatures_straightline_above_tcrit = np.linspace(np.min(ln_absolute_reduced_temperatures_above_tcrit), np.max(ln_absolute_reduced_temperatures_above_tcrit), num=1001)

    # Generating array of x coordinates for plotting straight line for below tcrit
    x_temperatures_straightline_below_tcrit = np.linspace(np.min(ln_absolute_reduced_temperatures_below_tcrit), np.max(ln_absolute_reduced_temperatures_below_tcrit), num=1001)

    ## Critical temperates overplotted on graphs of temperature vs magnetisation ##

    # Making plot of temperatures vs magnetisation - axvlines for the three methods and the onsager critical temperature
    fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
    ax.plot(run_temperatures_array, per_run_mean_equilibrated_magnetisations_per_spin_array, '-x')
    ax.axvline(x=tcrit_from_magnetisation, color='red', linestyle='--', label='tcrit magnetisation')
    ax.axvline(x=tcrit_from_magnetic_susceptibility, color='green', linestyle='--', label='tcrit magnetic susceptibility')
    ax.axvline(x=tcrit_from_heat_capacity, color='orange', linestyle='--', label='tcrit heat capacity')
    ax.axvline(x=average_tcrit, color='blue', linestyle='--', label='tcrit magnetisation, susceptibility and heat capacity average')
    ax.axvline(x=tcrit_onsager, color='grey', linestyle='--', label='tcrit Onsager 1944')
    ax.legend(loc='upper right')
    ax.grid(ls=':', color='0.9')
    ax.set_title('Magnetisation per spin vs temperature MMC')
    ax.set_xlabel('Temperature')
    ax.set_ylabel('Magnetisation per spin')

    # Saving figure
    fig.savefig(f'Temperature vs magnetisation, filename={filename}, MMC.png', dpi=300)

    # Making plot of temperatures vs heat capacity - axvlines for the three methods and the onsager critical temperature
    fig0, ax0 = plt.subplots(figsize=(6, 4), dpi=300)
    ax0.plot(run_temperatures_array, per_run_mean_equilibrated_heat_capacities_per_spin_array, '-x')
    ax0.axvline(x=tcrit_from_magnetisation, color='red', linestyle='--', label='tcrit magnetisation')
    ax0.axvline(x=tcrit_from_magnetic_susceptibility, color='green', linestyle='--', label='tcrit magnetic susceptibility')
    ax0.axvline(x=tcrit_from_heat_capacity, color='orange', linestyle='--', label='tcrit heat capacity')
    ax0.axvline(x=average_tcrit, color='blue', linestyle='--', label='tcrit magnetisation, susceptibility and heat capacity average')
    ax0.axvline(x=tcrit_onsager, color='grey', linestyle='--', label='tcrit Onsager 1944')
    ax0.legend(loc='upper right')
    ax0.grid(ls=':', color='0.9')
    ax0.set_title('Heat capacity per spin vs temperature MMC')
    ax0.set_xlabel('Temperature')
    ax0.set_ylabel('Heat capacity per spin')

    # Saving figure
    fig0.savefig(f'Temperature vs heat capacity, filename={filename}, MMC.png', dpi=300)

    # Making plot of temperature vs magnetic susceptibility axvlines for the three methods and the onsager critical temperature
    fig1, ax1 = plt.subplots(figsize=(6, 4), dpi=300)
    ax1.plot(run_temperatures_array, per_run_mean_equilibrated_magnetic_susceptibilities_per_spin_array, '-x')
    ax1.axvline(x=tcrit_from_magnetisation, color='red', linestyle='--', label='tcrit magnetisation')
    ax1.axvline(x=tcrit_from_magnetic_susceptibility, color='green', linestyle='--', label='tcrit magnetic susceptibility')
    ax1.axvline(x=tcrit_from_heat_capacity, color='orange', linestyle='--', label='tcrit heat capacity')
    ax1.axvline(x=average_tcrit, color='blue', linestyle='--', label='tcrit magnetisation, susceptibility and heat capacity average')
    ax1.axvline(x=tcrit_onsager, color='grey', linestyle='--', label='tcrit Onsager 1944')
    ax1.legend(loc='upper right')
    ax1.grid(ls=':', color='0.9')
    ax1.set_title('Magnetic susceptibility per spin vs temperature MMC')
    ax1.set_xlabel('Temperature')
    ax1.set_ylabel('Magnetic susceptibility per spin')

    # Saving figure
    fig1.savefig(f'Temperature vs magnetic susceptibility, filename={filename}, MMC.png', dpi=300)

    ## Error calculations for critical temperature ##
    
    # Getting absolute errors from the onsager critical temperature
    error_tcrit_magnetisation = np.abs(tcrit_onsager-tcrit_from_magnetisation)
    error_tcrit_magnetic_susceptibility = np.abs(tcrit_onsager-tcrit_from_magnetic_susceptibility)
    error_tcrit_heat_capacity = np.abs(tcrit_onsager-tcrit_from_heat_capacity)
    error_tcrit_average = np.abs(tcrit_onsager-average_tcrit)

    ## 5 Plots for the Onsager exponents in the valid regions - Corresponds to the 5 gradients ##

    # 1. Gradient plot for critical exponent alpha: ln(t - tcrit) vs ln(heat_capacity) above tcrit
    fig2, ax2 = plt.subplots(figsize=(6, 4), dpi=300)
    ax2.plot(ln_absolute_reduced_temperatures_above_tcrit, heat_capacities_above_tcrit, '-x')
    ax2.plot(x_temperatures_straightline_above_tcrit, straight_line(x_temperatures_straightline_above_tcrit, gradient_heat_capacity_above_tcrit, heat_capacity_intercept_above_tcrit), linestyle='--', color='black', label='Calculated Onsager Critical Amplitude above tcrit')
    ax2.plot(x_temperatures_straightline_above_tcrit, straight_line(x_temperatures_straightline_above_tcrit, negative_onsager_critical_exponent, heat_capacity_intercept_above_tcrit), linestyle='--', color='red', label='Onsager Critical Exponent')
    ax2.legend(loc='upper right')
    ax2.grid(ls=':', color='0.9')
    ax2.set_title('Onsager critical amplitude above tcrit MMC')
    ax2.set_xlabel('ln(|temps_above_tcrit - tcrit_average|)')
    ax2.set_ylabel('Heat Capacity')

    # Saving figure
    fig2.savefig(f'Onsager critical exponent above tcrit, filename={filename}, MMC.png', dpi=300)

    # 2. Gradient plot for critical exponent alpha: ln(t - tcrit) vs ln(heat_capacity) below tcrit
    fig3, ax3 = plt.subplots(figsize=(6, 4), dpi=300)
    ax3.plot(ln_absolute_reduced_temperatures_below_tcrit, heat_capacities_below_tcrit, '-x')
    ax3.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, gradient_heat_capacity_below_tcrit, heat_capacity_intercept_below_tcrit), linestyle='--', color='black', label='Calculated Onsager Critical Amplitude below tcrit')
    ax3.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, negative_onsager_critical_exponent, heat_capacity_intercept_below_tcrit), linestyle='--', color='red', label='Onsager Critical Exponent')
    ax3.legend(loc='upper right')
    ax3.grid(ls=':', color='0.9')
    ax3.set_title('Onsager critical amplitude below tcrit MMC')
    ax3.set_xlabel('ln(|temps_below_tcrit - tcrit_average|)')
    ax3.set_ylabel('Heat Capacity')

    # Saving figure
    fig3.savefig(f'Onsager critical exponent below tcrit, filename={filename}, MMC.png', dpi=300)

    # 3. Gradient plot for critical exponent beta: ln(t - tcrit) vs ln(magnetisation) below tcrit
    fig4, ax4 = plt.subplots(figsize=(6, 4), dpi=300)
    ax4.plot(ln_absolute_reduced_temperatures_below_tcrit, ln_magnetisations_below_tcrit, '-x')
    ax4.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, gradient_magnetisation_below_tcrit, magnetisation_intercept_below_tcrit), linestyle='--', color='black', label='Calculated Beta below tcrit')
    ax4.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, onsager_beta, magnetisation_intercept_below_tcrit), linestyle='--', color='red', label='Onsager Beta')
    ax4.legend(loc='upper right')
    ax4.grid(ls=':', color='0.9')
    ax4.set_title('Beta below tcrit MMC')
    ax4.set_xlabel('ln(|temps_below_tcrit - tcrit_average|)')
    ax4.set_ylabel('ln(|magnetisation per spin|)')

    # Saving figure
    fig4.savefig(f'Beta below tcrit, filename={filename}, MMC.png', dpi=300)

    # 4. Gradient plot for critical exponent Gamma: ln(t - tcrit) vs ln(heat_capacity) above tcrit
    fig5, ax5 = plt.subplots(figsize=(6, 4), dpi=300)
    ax5.plot(ln_absolute_reduced_temperatures_above_tcrit, ln_magnetic_susceptibilities_above_tcrit, '-x')
    ax5.plot(x_temperatures_straightline_above_tcrit, straight_line(x_temperatures_straightline_above_tcrit, gradient_magnetic_susceptibility_above_tcrit, magnetic_susceptibility_intercept_above_tcrit), linestyle='--', color='black', label='Calculated Gamma above tcrit')
    ax5.plot(x_temperatures_straightline_above_tcrit, straight_line(x_temperatures_straightline_above_tcrit, onsager_gamma, magnetic_susceptibility_intercept_above_tcrit), linestyle='--', color='red', label='Onsager Gamma')
    ax5.legend(loc='upper right')
    ax5.grid(ls=':', color='0.9')
    ax5.set_title('Gamma above tcrit MMC')
    ax5.set_xlabel('ln(|temps_above_tcrit - tcrit_average|)')
    ax5.set_ylabel('ln(|magnetic susceptibility per spin|)')

    # Saving figure
    fig5.savefig(f'Gamma above tcrit, filename={filename}, MMC.png', dpi=300)

    # 5. Gradient plot for critical exponent Gamma: ln(t - tcrit) vs ln(heat_capacity) below tcrit
    fig6, ax6 = plt.subplots(figsize=(6, 4), dpi=300)
    ax6.plot(ln_absolute_reduced_temperatures_below_tcrit, ln_magnetic_susceptibilities_below_tcrit, '-x')
    ax6.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, gradient_magnetic_susceptibility_below_tcrit, magnetic_susceptibility_intercept_below_tcrit), linestyle='--', color='black', label='Calculated Gamma below tcrit')
    ax6.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, onsager_gamma, magnetic_susceptibility_intercept_below_tcrit), linestyle='--', color='red', label='Onsager Gamma')
    ax6.legend(loc='upper right')
    ax6.grid(ls=':', color='0.9')
    ax6.set_title('Gamma below tcrit MMC')
    ax6.set_xlabel('ln(|temps_below_tcrit - tcrit_average|)')
    ax6.set_ylabel('ln(|magnetic susceptibility per spin|)')

    # Saving figure
    fig6.savefig(f'Gamma below tcrit, filename={filename}, MMC.png', dpi=300)

    ## Calculating average of the two gradients in each safety region for alpha and gamma. Beta doesnt need it as it is only valid below tcrit ##

    average_negative_onsager_critical_exponent = (gradient_heat_capacity_above_tcrit + gradient_heat_capacity_below_tcrit) / 2
    average_gamma = (gradient_magnetic_susceptibility_above_tcrit + gradient_magnetic_susceptibility_below_tcrit) / 2

    ## Calculating errors in critical exponents for above and below tcrit ##

    # Errors for alpha in both regimes and the average
    error_negative_onsager_critical_exponent_above_tcrit = np.abs(negative_onsager_critical_exponent - gradient_heat_capacity_above_tcrit)
    error_negative_onsager_critical_exponent_below_tcrit = np.abs(negative_onsager_critical_exponent - gradient_heat_capacity_below_tcrit)
    error_average_negative_onsager_critical_exponent = np.abs(negative_onsager_critical_exponent - average_negative_onsager_critical_exponent)

    # Error in beta below tcrit 
    error_beta = np.abs(onsager_beta-gradient_magnetisation_below_tcrit)

    # Errors for alpha in both regimes and the average
    error_gamma_above_tcrit = np.abs(onsager_gamma - gradient_magnetic_susceptibility_above_tcrit)
    error_gamma_below_tcrit = np.abs(onsager_gamma - gradient_magnetic_susceptibility_below_tcrit)
    error_average_gamma = np.abs(onsager_gamma - average_gamma)
    
    ## Printing out critical temperature results so it is easy to access them ##

    # Critical temperature results
    print('*** Critical Temperature Results ***')
    print(f'The Onsager 1944 result for the critical temperature of an infinite lattice is {tcrit_onsager}K.')
    print(f'Calculated critical temperature from magnetisation drop using the metropolis algorithm is {tcrit_from_magnetisation}K.')
    print(f'Calculated critical temperature from magnetic susceptibility peak using the metropolis algorithm is {tcrit_from_magnetic_susceptibility}K.')
    print(f'Calculated critical temperature from heat capacity peak using the metropolis algorithm is {tcrit_from_heat_capacity}K.')
    print(f'Average from the three methods for calculating critical temperature using the metropolis algoritm is {average_tcrit}K.')
    print('\n')

    print('*** Critical Temperature Errors ***')
    print(f'Error in critical temperature for the magnetisation drop method using the metropolis algorithm is {error_tcrit_magnetisation}K.')
    print(f'Error in critical temperature for the magnetic susceptibility peak method using the metropolis algorithm is {error_tcrit_magnetic_susceptibility}K.')
    print(f'Error in critical temperature from heat capacity peak using the metropolis algorithm is {error_tcrit_heat_capacity}K')
    print(f'Error in critical temperature for the average between the three methods using the metropolis algorithm is {error_tcrit_average}.')
    print('\n')

    ## Printing out critical exponent results and errors so it is easy to access them ##

    # Onsager values for critical exponents
    print('*** Onsager Critical Exponents ***')
    print(f'The Onsager 1944 result for the critical amplitude {np.abs(negative_onsager_critical_exponent)}.')
    print('The Onsager 1944 result for the critical exponent alpha is 0')
    print(f'The Onsager 1944 result for the critical exponent beta is {onsager_beta}.')
    print(f'The Onsager 1944 result for the critical exponent gamma is {onsager_gamma}.')
    print('\n')

    # Results for Onsager critical amplitude
    print('Since alpha is 0, the best way to verify it is a linear plot of ln(T) vs the heat capacity.')
    print('If the simulation is working correctly and alpha is behaving as Onsager predicted, then the plot will be linear and have a gradient of -0.4945, which is the negative of the Onsager critical amplitude.')
    print('\n')
    print('*** Results for Onsager Critical Amplitude ***')
    print(f'The calculated value of the negative Onsager critical amplitude above tcrit is {gradient_heat_capacity_above_tcrit}.')
    print(f'The calculated value of the negative Onsager critical amplitude below tcrit is {gradient_heat_capacity_below_tcrit}.')
    print(f'The calculated value of the negative Onsager critical amplitude using the average of the two temperature regimes is {average_negative_onsager_critical_exponent}.')
    print('\n')

    # Errors for alpha in both temperature regimes and the average value
    print('*** Errors for Onsager critical amplitude ***')
    print(f'Error in Onsager critical amplitude above tcrit is {error_negative_onsager_critical_exponent_above_tcrit}')
    print(f'Error in Onsager critical amplitude below tcrit is {error_negative_onsager_critical_exponent_below_tcrit}')
    print(f'Error in Onsager critical amplitude for the average of both temperature regimes is {error_average_negative_onsager_critical_exponent}')
    print('\n')
    
    # Result for beta
    print('*** Results for exponent beta ***')
    print(f'The calculated value of beta below tcrit is {gradient_magnetisation_below_tcrit}')
    print('\n')

    # Error for beta
    print('*** Error for exponent beta ***')
    print(f'Error in beta for below tcrit is {error_beta}')
    print('\n')

    # Results for gamma
    print('*** Results for exponent gamma ***')
    print(f'The calculated value of gamma above tcrit is {gradient_magnetic_susceptibility_above_tcrit}.')
    print(f'The calculated value of gamma below tcrit is {gradient_magnetic_susceptibility_below_tcrit}.')
    print(f'The calculated value of gamma using the average of the two temperature regimes is {average_gamma}.')
    print('\n')

    # Errors for gamma in both temperature regimes and the average value
    print('*** Errors for exponent gamma ***')
    print(f'Error in gamma above tcrit is {error_gamma_above_tcrit}')
    print(f'Error in gamma below tcrit is {error_gamma_below_tcrit}')
    print(f'Error in gamma for the average of both temperature regimes is {error_average_gamma}')
    print('\n')
    
    # This will be used on the validation plots to conserve memory
    if plot_mode == 'close':
        plt.close('all')
            
    # If no mode is defined, force the plots to display
    elif plot_mode == None:
        plt.show('all')
    
    return fig, ax, fig0, ax0, fig1, ax1, fig2, ax2, fig3, ax3, fig4, ax4, fig5, ax5, fig6, ax6

# Function for running parallelised critical temperature and exponents investigation using the wolff algorithm

In [19]:
def run_parallelised_temp_change_investigation_simulations_WMC(parameter_file, **kwargs):

    """Calls the run_single_temp_investigation_simulation(args) function N cores times per instance and packages the results up into lists.
    Once enough instances have been called that the full temperature sweep has been completed, the lists of mean magnetisations, 
    mean magnetic susceptibilities and mean heat capacities are converted into arrays and used for critical temperature and critical 
    exponent calculations.
    
    In order to run the run_single_temp_investigation_simulation(args) function in parallel, it uses all available cpu cores and the pool module 
    from multiprocessing package.
    
    Returns plots of magnetisations and magnetic susceptibility vs temperatures."""

    # Getting the plot mode - condition to show / hide the diagnostic plots
    plot_mode = kwargs.get('plot_mode', None)

    # Setting up ICS and the lattice from the file 
    lattice_side_length, magnetic_field_B, initial_temperature, temp_step, exchange_constant_J, k_b, u_b, nsteps, nruns, filename = read_tcrit_investigative_file(parameter_file)

    # Making a list to store temperatures
    run_temperatures_list = []

    # Appending to the list using temp step and the amount of runs
    for i in range(nruns):

        run_temperatures_list.append(initial_temperature + i*temp_step)

    # Turning the list into an array as this will be useful later
    run_temperatures_array = np.asarray(run_temperatures_list)

    ## Setting up parallelised simulation runs ##

    # Runs for worker function defined in the file 
    single_temp_run_args = [(temperature, lattice_side_length, magnetic_field_B, exchange_constant_J, k_b, u_b, nsteps) for temperature in run_temperatures_array]

    # Printing out initialisation message for parallelised simulations
    print(f'Parallelising {nruns} temperature investigation runs across {n_cores} CPU cores using the wolff algorithm.')
    print(f'Temperature range: {run_temperatures_array[0]} --> {run_temperatures_array[-1]}.')
    print(f'Per-simulation parameters: temp_step={temp_step}K, lattice_side_length={lattice_side_length}, magnetic_field_B={magnetic_field_B}, exchange_constant_J={exchange_constant_J}, k_b={k_b}, u_b={u_b}, nsteps={nsteps}.')

    # Running multiple simulations in parallel
    with Pool(processes=os.cpu_count()) as pool:
        
        # Calling the imported parallelisable worker function to run multiple simulations at the same time using multiprocessing
        # This completes the runs across multiple temperatures much faster than using one core only
        per_run_mean_quantites = list(tqdm(pool.imap(run_WMC_temp_investigation_worker, single_temp_run_args), total=nruns, desc='Running parallelised wolff algorithm temperature sweep:'))

    # Unpacking results from the list generated using multiprocessing
    # This has to be done in the same order as the run_single_temp_investigation_simulation(args) returns them in.
    per_run_mean_equilibrated_magnetisations_per_spin_array = np.array([mean_results[0] for mean_results in per_run_mean_quantites])
    per_run_mean_equilibrated_magnetic_susceptibilities_per_spin_array = np.array([mean_results[1] for mean_results in per_run_mean_quantites])
    per_run_mean_equilibrated_heat_capacities_per_spin_array = np.array([mean_results[2] for mean_results in per_run_mean_quantites])
    
    ## Critical temperature extraction ##
    
    # Method 1: Steepest drop in absolute magnetisation - Midpoint of the two temperatures between where this happens should yield phase transition temperature.
    consecutive_magnetisation_diffs = np.diff(per_run_mean_equilibrated_magnetisations_per_spin_array)
    largest_magnetisation_drop_index = np.argmin(consecutive_magnetisation_diffs)
    
    # Average of 2 values before and after the index for the most negative drop in magnetisation should yield the critical temperature
    tcrit_from_magnetisation = (run_temperatures_array[largest_magnetisation_drop_index] + run_temperatures_array[largest_magnetisation_drop_index+1])/2
    
    # Method 2: Index of the temperature where magnetic susceptibility peaks
    largest_magnetic_susceptibility_index = np.argmax(per_run_mean_equilibrated_magnetic_susceptibilities_per_spin_array)
    tcrit_from_magnetic_susceptibility = run_temperatures_array[largest_magnetic_susceptibility_index]

    # Method 3: Index of temperature where heat capacity peaks
    largest_heat_capacity_index = np.argmax(per_run_mean_equilibrated_heat_capacities_per_spin_array)
    tcrit_from_heat_capacity = run_temperatures_array[largest_heat_capacity_index]

    # Method 3: Average of methods 1, 2 and 3 - Most statistical accuracy
    average_tcrit = (tcrit_from_magnetisation + tcrit_from_magnetic_susceptibility + tcrit_from_heat_capacity)/3

    ## Storage of Onsager's calculated critical temperature and critical exponents for comparison and error calculations ##

    # Onsager 1944 critical temperature
    tcrit_onsager = 2.269185

    # Storing Onsager 1944 critical exponents 

    # Alpha critical exponent
    negative_onsager_critical_exponent = -0.4945

    # Beta critical exponent
    onsager_beta = 1/8

    # Gamma critical exponent
    onsager_gamma = -7/4

    ## Temperature regime filters ##

    # Above tcrit filter with 1% safety threshold.
    above_tcrit_filter = run_temperatures_array > average_tcrit*1.001

    # Below tcrit filter with 1% safety threshold.
    below_tcrit_filter = run_temperatures_array < average_tcrit*0.99

    ## Using reduced temperatures because they are dimensionless and thus ideal for power law scaling ##
    
    # Reduced temperatures using data above tcrit
    absolute_reduced_temperatures_above_tcrit = (run_temperatures_array[above_tcrit_filter] - average_tcrit) / average_tcrit

    # Reduced temperatures using data below tcrit
    absolute_reduced_temperatures_below_tcrit = (run_temperatures_array[below_tcrit_filter] - average_tcrit) / average_tcrit

    ## Heat capacity, magnetisation and susceptibility filtering ##

    # Separate heat capacities above and below tcrit
    heat_capacities_above_tcrit = per_run_mean_equilibrated_heat_capacities_per_spin_array[above_tcrit_filter]
    heat_capacities_below_tcrit = per_run_mean_equilibrated_heat_capacities_per_spin_array[below_tcrit_filter]

    # Separate magnetisations below tcrit. Only applies below tcrit
    magnetisations_below_tcrit = per_run_mean_equilibrated_magnetisations_per_spin_array[below_tcrit_filter]

    # Separate magnetic susceptibilities above and below tcrit
    magnetic_susceptibilities_above_tcrit = per_run_mean_equilibrated_magnetic_susceptibilities_per_spin_array[above_tcrit_filter]
    magnetic_susceptibilities_below_tcrit = per_run_mean_equilibrated_magnetic_susceptibilities_per_spin_array[below_tcrit_filter]

    ## Computing natural logarithms of absolute reduced temperatures above and below tcrit ##

    # np.abs is used to prevent the logarithm of a negative quantity from being computed. 
    ln_absolute_reduced_temperatures_above_tcrit = np.log(np.abs(absolute_reduced_temperatures_above_tcrit))
    ln_absolute_reduced_temperatures_below_tcrit = np.log(np.abs(absolute_reduced_temperatures_below_tcrit))

    ## Computing natural logarithm of magnetisation below tcrit

    # np.abs is used to prevent the logarithm of a negative quantity from being computed.
    ln_magnetisations_below_tcrit = np.log(np.abs(magnetisations_below_tcrit))

    ## Computing natural logarithms of magnetic susceptibilities above and below tcrit ##

    # np.abs is used to prevent the logarithm of a negative quantity from being computed.
    ln_magnetic_susceptibilities_above_tcrit = np.log(np.abs(magnetic_susceptibilities_above_tcrit))
    ln_magnetic_susceptibilities_below_tcrit = np.log(np.abs(magnetic_susceptibilities_below_tcrit))

    ## Computing the 5 different gradients ##

    # Gradient of ln heat capacity vs ln reduced temperatures above and below tcrit
    gradient_heat_capacity_above_tcrit, error_gradient_heat_capacity_above_tcrit, heat_capacity_intercept_above_tcrit, error_heat_capacity_intercept_above_tcrit = calculate_linear_regress_gradient_fit(ln_absolute_reduced_temperatures_above_tcrit, heat_capacities_above_tcrit)
    gradient_heat_capacity_below_tcrit, error_gradient_heat_capacity_below_tcrit, heat_capacity_intercept_below_tcrit, error_heat_capacity_intercept_below_tcrit = calculate_linear_regress_gradient_fit(ln_absolute_reduced_temperatures_below_tcrit, heat_capacities_below_tcrit)

    # Gradient of ln magnetisation vs ln reduced temperatures above and below tcrit
    gradient_magnetisation_below_tcrit, error_gradient_heat_capacity_below_tcrit, magnetisation_intercept_below_tcrit, error_magnetisation_intercept_below_tcrit = calculate_linear_regress_gradient_fit(ln_absolute_reduced_temperatures_below_tcrit, ln_magnetisations_below_tcrit)

    # Gradient of ln magnetic susceptibility vs ln reduced temperatures above and below tcrit
    gradient_magnetic_susceptibility_above_tcrit, error_gradient_magnetic_susceptibility_above_tcrit, magnetic_susceptibility_intercept_above_tcrit, error_magnetic_susceptibility_intercept_above_tcrit = calculate_linear_regress_gradient_fit(ln_absolute_reduced_temperatures_above_tcrit, ln_magnetic_susceptibilities_above_tcrit)
    gradient_magnetic_susceptibility_below_tcrit, error_gradient_magnetic_susceptibility_below_tcrit, magnetic_susceptibility_intercept_below_tcrit, error_magnetic_susceptibility_intercept_below_tcrit = calculate_linear_regress_gradient_fit(ln_absolute_reduced_temperatures_below_tcrit, ln_magnetic_susceptibilities_below_tcrit)

    ### Making plots ###

    # Generating array of x coordinates for plotting straight line for above tcrit
    x_temperatures_straightline_above_tcrit = np.linspace(np.min(ln_absolute_reduced_temperatures_above_tcrit), np.max(ln_absolute_reduced_temperatures_above_tcrit), num=1001)

    # Generating array of x coordinates for plotting straight line for below tcrit
    x_temperatures_straightline_below_tcrit = np.linspace(np.min(ln_absolute_reduced_temperatures_below_tcrit), np.max(ln_absolute_reduced_temperatures_below_tcrit), num=1001)

    ## Critical temperates overplotted on graphs of temperature vs magnetisation ##

    # Making plot of temperatures vs magnetisation - axvlines for the three methods and the onsager critical temperature
    fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
    ax.plot(run_temperatures_array, per_run_mean_equilibrated_magnetisations_per_spin_array, '-x')
    ax.axvline(x=tcrit_from_magnetisation, color='red', linestyle='--', label='tcrit magnetisation')
    ax.axvline(x=tcrit_from_magnetic_susceptibility, color='green', linestyle='--', label='tcrit magnetic susceptibility')
    ax.axvline(x=tcrit_from_heat_capacity, color='orange', linestyle='--', label='tcrit heat capacity')
    ax.axvline(x=average_tcrit, color='blue', linestyle='--', label='tcrit magnetisation, susceptibility and heat capacity average')
    ax.axvline(x=tcrit_onsager, color='grey', linestyle='--', label='tcrit Onsager 1944')
    ax.legend(loc='upper right')
    ax.grid(ls=':', color='0.9')
    ax.set_title('Magnetisation per spin vs temperature WMC')
    ax.set_xlabel('Temperature')
    ax.set_ylabel('Magnetisation per spin')

    # Saving figure
    fig.savefig(f'Temperature vs magnetisation, filename={filename}, WMC.png', dpi=300)

    # Making plot of temperatures vs heat capacity - axvlines for the three methods and the onsager critical temperature
    fig0, ax0 = plt.subplots(figsize=(6, 4), dpi=300)
    ax0.plot(run_temperatures_array, per_run_mean_equilibrated_heat_capacities_per_spin_array, '-x')
    ax0.axvline(x=tcrit_from_magnetisation, color='red', linestyle='--', label='tcrit magnetisation')
    ax0.axvline(x=tcrit_from_magnetic_susceptibility, color='green', linestyle='--', label='tcrit magnetic susceptibility')
    ax0.axvline(x=tcrit_from_heat_capacity, color='orange', linestyle='--', label='tcrit heat capacity')
    ax0.axvline(x=average_tcrit, color='blue', linestyle='--', label='tcrit magnetisation, susceptibility and heat capacity average')
    ax0.axvline(x=tcrit_onsager, color='grey', linestyle='--', label='tcrit Onsager 1944')
    ax0.legend(loc='upper right')
    ax0.grid(ls=':', color='0.9')
    ax0.set_title('Heat capacity per spin vs temperature WMC')
    ax0.set_xlabel('Temperature')
    ax0.set_ylabel('Heat capacity per spin')

    # Saving figure
    fig0.savefig(f'Temperature vs heat capacity, filename={filename}, WMC.png', dpi=300)

    # Making plot of temperature vs magnetic susceptibility axvlines for the three methods and the onsager critical temperature
    fig1, ax1 = plt.subplots(figsize=(6, 4), dpi=300)
    ax1.plot(run_temperatures_array, per_run_mean_equilibrated_magnetic_susceptibilities_per_spin_array, '-x')
    ax1.axvline(x=tcrit_from_magnetisation, color='red', linestyle='--', label='tcrit magnetisation')
    ax1.axvline(x=tcrit_from_magnetic_susceptibility, color='green', linestyle='--', label='tcrit magnetic susceptibility')
    ax1.axvline(x=tcrit_from_heat_capacity, color='orange', linestyle='--', label='tcrit heat capacity')
    ax1.axvline(x=average_tcrit, color='blue', linestyle='--', label='tcrit magnetisation, susceptibility and heat capacity average')
    ax1.axvline(x=tcrit_onsager, color='grey', linestyle='--', label='tcrit Onsager 1944')
    ax1.legend(loc='upper right')
    ax1.grid(ls=':', color='0.9')
    ax1.set_title('Magnetic susceptibility per spin vs temperature WMC')
    ax1.set_xlabel('Temperature')
    ax1.set_ylabel('Magnetic susceptibility per spin')

    # Saving figure
    fig1.savefig(f'Temperature vs magnetic susceptibility, filename={filename}, WMC.png', dpi=300)

    ## Error calculations for critical temperature ##
    
    # Getting absolute errors from the onsager critical temperature
    error_tcrit_magnetisation = np.abs(tcrit_onsager-tcrit_from_magnetisation)
    error_tcrit_magnetic_susceptibility = np.abs(tcrit_onsager-tcrit_from_magnetic_susceptibility)
    error_tcrit_heat_capacity = np.abs(tcrit_onsager-tcrit_from_heat_capacity)
    error_tcrit_average = np.abs(tcrit_onsager-average_tcrit)

    ## 5 Plots for the Onsager exponents in the valid regions - Corresponds to the 5 gradients ##

    # 1. Gradient plot for critical exponent alpha: ln(t - tcrit) vs ln(heat_capacity) above tcrit
    fig2, ax2 = plt.subplots(figsize=(6, 4), dpi=300)
    ax2.plot(ln_absolute_reduced_temperatures_above_tcrit, heat_capacities_above_tcrit, '-x')
    ax2.plot(x_temperatures_straightline_above_tcrit, straight_line(x_temperatures_straightline_above_tcrit, gradient_heat_capacity_above_tcrit, heat_capacity_intercept_above_tcrit), linestyle='--', color='black', label='Calculated Onsager Critical Amplitude above tcrit')
    ax2.plot(x_temperatures_straightline_above_tcrit, straight_line(x_temperatures_straightline_above_tcrit, negative_onsager_critical_exponent, heat_capacity_intercept_above_tcrit), linestyle='--', color='red', label='Onsager Critical Exponent')
    ax2.legend(loc='upper right')
    ax2.grid(ls=':', color='0.9')
    ax2.set_title('Onsager critical amplitude above tcrit WMC')
    ax2.set_xlabel('ln(|temps_above_tcrit - tcrit_average|)')
    ax2.set_ylabel('Heat Capacity')

    # Saving figure
    fig2.savefig(f'Onsager critical exponent above tcrit, filename={filename}, WMC.png', dpi=300)

    # 2. Gradient plot for critical exponent alpha: ln(t - tcrit) vs ln(heat_capacity) below tcrit
    fig3, ax3 = plt.subplots(figsize=(6, 4), dpi=300)
    ax3.plot(ln_absolute_reduced_temperatures_below_tcrit, heat_capacities_below_tcrit, '-x')
    ax3.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, gradient_heat_capacity_below_tcrit, heat_capacity_intercept_below_tcrit), linestyle='--', color='black', label='Calculated Onsager Critical Amplitude below tcrit')
    ax3.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, negative_onsager_critical_exponent, heat_capacity_intercept_below_tcrit), linestyle='--', color='red', label='Onsager Critical Exponent')
    ax3.legend(loc='upper right')
    ax3.grid(ls=':', color='0.9')
    ax3.set_title('Onsager critical amplitude below tcrit WMC')
    ax3.set_xlabel('ln(|temps_below_tcrit - tcrit_average|)')
    ax3.set_ylabel('Heat Capacity')

    # Saving figure
    fig3.savefig(f'Onsager critical exponent below tcrit, filename={filename}, WMC.png', dpi=300)

    # 3. Gradient plot for critical exponent beta: ln(t - tcrit) vs ln(magnetisation) below tcrit
    fig4, ax4 = plt.subplots(figsize=(6, 4), dpi=300)
    ax4.plot(ln_absolute_reduced_temperatures_below_tcrit, ln_magnetisations_below_tcrit, '-x')
    ax4.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, gradient_magnetisation_below_tcrit, magnetisation_intercept_below_tcrit), linestyle='--', color='black', label='Calculated Beta below tcrit')
    ax4.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, onsager_beta, magnetisation_intercept_below_tcrit), linestyle='--', color='red', label='Onsager Beta')
    ax4.legend(loc='upper right')
    ax4.grid(ls=':', color='0.9')
    ax4.set_title('Beta below tcrit WMC')
    ax4.set_xlabel('ln(|temps_below_tcrit - tcrit_average|)')
    ax4.set_ylabel('ln(|magnetisation per spin|)')

    # Saving figure
    fig4.savefig(f'Beta below tcrit, filename={filename}, WMC.png', dpi=300)

    # 4. Gradient plot for critical exponent Gamma: ln(t - tcrit) vs ln(heat_capacity) above tcrit
    fig5, ax5 = plt.subplots(figsize=(6, 4), dpi=300)
    ax5.plot(ln_absolute_reduced_temperatures_above_tcrit, ln_magnetic_susceptibilities_above_tcrit, '-x')
    ax5.plot(x_temperatures_straightline_above_tcrit, straight_line(x_temperatures_straightline_above_tcrit, gradient_magnetic_susceptibility_above_tcrit, magnetic_susceptibility_intercept_above_tcrit), linestyle='--', color='black', label='Calculated Gamma above tcrit')
    ax5.plot(x_temperatures_straightline_above_tcrit, straight_line(x_temperatures_straightline_above_tcrit, onsager_gamma, magnetic_susceptibility_intercept_above_tcrit), linestyle='--', color='red', label='Onsager Gamma')
    ax5.legend(loc='upper right')
    ax5.grid(ls=':', color='0.9')
    ax5.set_title('Gamma above tcrit WMC')
    ax5.set_xlabel('ln(|temps_above_tcrit - tcrit_average|)')
    ax5.set_ylabel('ln(|magnetic susceptibility per spin|)')

    # Saving figure
    fig5.savefig(f'Gamma above tcrit, filename={filename}, WMC.png', dpi=300)

    # 5. Gradient plot for critical exponent Gamma: ln(t - tcrit) vs ln(heat_capacity) below tcrit
    fig6, ax6 = plt.subplots(figsize=(6, 4), dpi=300)
    ax6.plot(ln_absolute_reduced_temperatures_below_tcrit, ln_magnetic_susceptibilities_below_tcrit, '-x')
    ax6.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, gradient_magnetic_susceptibility_below_tcrit, magnetic_susceptibility_intercept_below_tcrit), linestyle='--', color='black', label='Calculated Gamma below tcrit')
    ax6.plot(x_temperatures_straightline_below_tcrit, straight_line(x_temperatures_straightline_below_tcrit, onsager_gamma, magnetic_susceptibility_intercept_below_tcrit), linestyle='--', color='red', label='Onsager Gamma')
    ax6.legend(loc='upper right')
    ax6.grid(ls=':', color='0.9')
    ax6.set_title('Gamma below tcrit WMC')
    ax6.set_xlabel('ln(|temps_below_tcrit - tcrit_average|)')
    ax6.set_ylabel('ln(|magnetic susceptibility per spin|)')

    # Saving figure
    fig6.savefig(f'Gamma below tcrit, filename={filename}, WMC.png', dpi=300)

    ## Calculating average of the two gradients in each safety region for alpha and gamma. Beta doesnt need it as it is only valid below tcrit ##

    average_negative_onsager_critical_exponent = (gradient_heat_capacity_above_tcrit + gradient_heat_capacity_below_tcrit) / 2
    average_gamma = (gradient_magnetic_susceptibility_above_tcrit + gradient_magnetic_susceptibility_below_tcrit) / 2

    ## Calculating errors in critical exponents for above and below tcrit ##

    # Errors for alpha in both regimes and the average
    error_negative_onsager_critical_exponent_above_tcrit = np.abs(negative_onsager_critical_exponent - gradient_heat_capacity_above_tcrit)
    error_negative_onsager_critical_exponent_below_tcrit = np.abs(negative_onsager_critical_exponent - gradient_heat_capacity_below_tcrit)
    error_average_negative_onsager_critical_exponent = np.abs(negative_onsager_critical_exponent - average_negative_onsager_critical_exponent)

    # Error in beta below tcrit 
    error_beta = np.abs(onsager_beta-gradient_magnetisation_below_tcrit)

    # Errors for alpha in both regimes and the average
    error_gamma_above_tcrit = np.abs(onsager_gamma - gradient_magnetic_susceptibility_above_tcrit)
    error_gamma_below_tcrit = np.abs(onsager_gamma - gradient_magnetic_susceptibility_below_tcrit)
    error_average_gamma = np.abs(onsager_gamma - average_gamma)
    
    ## Printing out critical temperature results so it is easy to access them ##

    # Critical temperature results
    print('*** Critical Temperature Results ***')
    print(f'The Onsager 1944 result for the critical temperature of an infinite lattice is {tcrit_onsager}K.')
    print(f'Calculated critical temperature from magnetisation drop using the wolff algorithm is {tcrit_from_magnetisation}K.')
    print(f'Calculated critical temperature from magnetic susceptibility peak using the wolff algorithm is {tcrit_from_magnetic_susceptibility}K.')
    print(f'Calculated critical temperature from heat capacity peak using the wolff algorithm is {tcrit_from_heat_capacity}K.')
    print(f'Average from the three methods for calculating critical temperature using the wolff algoritm is {average_tcrit}K.')
    print('\n')

    print('*** Critical Temperature Errors ***')
    print(f'Error in critical temperature for the magnetisation drop method using the wolff algorithm is {error_tcrit_magnetisation}K.')
    print(f'Error in critical temperature for the magnetic susceptibility peak method using the wolff algorithm is {error_tcrit_magnetic_susceptibility}K.')
    print(f'Error in critical temperature from heat capacity peak using the wolff algorithm is {error_tcrit_heat_capacity}K')
    print(f'Error in critical temperature for the average between the three methods using the wolff algorithm is {error_tcrit_average}.')
    print('\n')

    ## Printing out critical exponent results and errors so it is easy to access them ##

    # Onsager values for critical exponents
    print('*** Onsager Critical Exponents ***')
    print(f'The Onsager 1944 result for the critical amplitude {np.abs(negative_onsager_critical_exponent)}.')
    print('The Onsager 1944 result for the critical exponent alpha is 0')
    print(f'The Onsager 1944 result for the critical exponent beta is {onsager_beta}.')
    print(f'The Onsager 1944 result for the critical exponent gamma is {onsager_gamma}.')
    print('\n')

    # Results for Onsager critical amplitude
    print('Since alpha is 0, the best way to verify it is a linear plot of ln(T) vs the heat capacity.')
    print('If the simulation is working correctly and alpha is behaving as Onsager predicted, then the plot will be linear and have a gradient of -0.4945, which is the negative of the Onsager critical amplitude.')
    print('\n')
    print('*** Results for Onsager Critical Amplitude ***')
    print(f'The calculated value of the negative Onsager critical amplitude above tcrit is {gradient_heat_capacity_above_tcrit}.')
    print(f'The calculated value of the negative Onsager critical amplitude below tcrit is {gradient_heat_capacity_below_tcrit}.')
    print(f'The calculated value of the negative Onsager critical amplitude using the average of the two temperature regimes is {average_negative_onsager_critical_exponent}.')
    print('\n')

    # Errors for alpha in both temperature regimes and the average value
    print('*** Errors for Onsager critical amplitude ***')
    print(f'Error in Onsager critical amplitude above tcrit is {error_negative_onsager_critical_exponent_above_tcrit}')
    print(f'Error in Onsager critical amplitude below tcrit is {error_negative_onsager_critical_exponent_below_tcrit}')
    print(f'Error in Onsager critical amplitude for the average of both temperature regimes is {error_average_negative_onsager_critical_exponent}')
    print('\n')
    
    # Result for beta
    print('*** Results for exponent beta ***')
    print(f'The calculated value of beta below tcrit is {gradient_magnetisation_below_tcrit}')
    print('\n')

    # Error for beta
    print('*** Error for exponent beta ***')
    print(f'Error in beta for below tcrit is {error_beta}')
    print('\n')

    # Results for gamma
    print('*** Results for exponent gamma ***')
    print(f'The calculated value of gamma above tcrit is {gradient_magnetic_susceptibility_above_tcrit}.')
    print(f'The calculated value of gamma below tcrit is {gradient_magnetic_susceptibility_below_tcrit}.')
    print(f'The calculated value of gamma using the average of the two temperature regimes is {average_gamma}.')
    print('\n')

    # Errors for gamma in both temperature regimes and the average value
    print('*** Errors for exponent gamma ***')
    print(f'Error in gamma above tcrit is {error_gamma_above_tcrit}')
    print(f'Error in gamma below tcrit is {error_gamma_below_tcrit}')
    print(f'Error in gamma for the average of both temperature regimes is {error_average_gamma}')
    print('\n')
    
    # This will be used on the validation plots to conserve memory
    if plot_mode == 'close':
        plt.close('all')
            
    # If no mode is defined, force the plots to display
    elif plot_mode == None:
        plt.show('all')
    
    return fig, ax, fig0, ax0, fig1, ax1, fig2, ax2, fig3, ax3, fig4, ax4, fig5, ax5, fig6, ax6

# Function to run parallelised lattice side length change investigation (calculating the Onsager critical exponent from lattice size changes) using the metropolis algorithm

In [20]:
def run_parallelised_lattice_side_length_change_investigation_simulations_MMC(parameter_file, **kwargs):

    """Calls the run_single_temp_investigation_simulation(args) function N cores times per instance and packages the results up into lists.
    Once enough instances have been called that the full temperature sweep has been completed, the lists of median magnetisations, 
    median magnetic susceptibilities and median heat capacities are converted into arrays and used for critical temperature and critical 
    exponent calculations.
    
    In order to run the run_single_temp_investigation_simulation(args) function in parallel, it uses all available cpu cores and the pool module 
    from multiprocessing package.
    
    Returns plots of magnetisations and magnetic susceptibility vs temperatures."""

    # Getting the plot mode - condition to show / hide the diagnostic plots
    plot_mode = kwargs.get('plot_mode', None)

    # Setting up ICS and the lattice from the file 
    initial_lattice_side_length, magnetic_field_B, temperature, lattice_side_length_step, exchange_constant_J, k_b, u_b, nsteps, nruns, filename = read_lattice_side_length_investigative_file(parameter_file)

    # Making a list to store temperatures
    run_lattice_side_lenths_list = []

    # Appending to the list using lattice side length step and the amount of runs
    for i in range(nruns):

        run_lattice_side_lenths_list.append(initial_lattice_side_length + i*lattice_side_length_step)

    # Turning the list into an array as this will be useful later
    run_lattice_side_lengths_array = np.asarray(run_lattice_side_lenths_list)

    ## Setting up parallelised simulation runs ##

    # Runs for worker function defined in the file 
    single_lattice_size_run_args = [(temperature, lattice_side_length, magnetic_field_B, exchange_constant_J, k_b, u_b, nsteps) for lattice_side_length in run_lattice_side_lengths_array]

    # Printing out initialisation message for parallelised simulations
    print(f'Parallelising {nruns} lattice size runs across {n_cores} CPU cores using the metropolis algorithm.')
    print(f'Lattice size range: {(run_lattice_side_lengths_array[0])**2} --> {(run_lattice_side_lengths_array[-1])**2}.')
    print(f'Per-simulation parameters: temperature={temperature}, lattice side length step={lattice_side_length_step}, magnetic_field_B={magnetic_field_B}, exchange_constant_J={exchange_constant_J}, k_b={k_b}, u_b={u_b}, nsteps={nsteps}')

    # Running multiple simulations in parallel
    with Pool(processes=os.cpu_count()) as pool:
        
        # Calling the imported parallelisable worker function to run multiple simulations at the same time using multiprocessing
        # This completes the runs across multiple temperatures much faster than using one core only
        per_run_median_quantites = list(tqdm(pool.imap(run_MMC_lattice_side_length_investigation_worker, single_lattice_size_run_args), total=nruns, desc='Running parallelised metropolis algorithm lattice size sweep:'))

    # Unpacking results from the list generated using multiprocessing
    # This has to be done in the same order as the run_single_temp_investigation_simulation(args) returns them in.
    # No need for indexing because the worker function for lattice side length only returns one quantity - The heat capacity
    per_run_median_equilibrated_heat_capacities_per_spin_array = np.array([median_results for median_results in per_run_median_quantites])

    # Computing the lattice sizes
    lattice_size_array = np.square(run_lattice_side_lengths_array)

    # Computing the per element logairthm of the lattice sizes
    ln_lattice_size_array = np.log(lattice_size_array)

    # Calculating C_0
    gradient_C_0, error_gradient_C_0, C_0_intercept, error_C_0_intercept = calculate_linear_regress_gradient_fit(ln_lattice_size_array, per_run_median_equilibrated_heat_capacities_per_spin_array)

    # Storing Onsager's 1944 value of C_0 - Half of the critical amplitude
    onsager_C_0 = 0.24725

    ## Error Calculation ##
    error_C_0 = np.abs(gradient_C_0 - onsager_C_0)

    # Generating array of x coordinates for plotting straight line
    x_lattice_sizes_straightline = np.linspace(np.min(ln_lattice_size_array), np.max(ln_lattice_size_array), num=1001)

    # Printing out the results for the calculated C_0 and its error compared to the onsager value.
    print('*** C_0 results ***')
    print(f'Calculated value of C_0 is {gradient_C_0}.')
    print(f'Onsager value of C_0 is {onsager_C_0}.')
    print(f'Error in calculated value of C_0 is {error_C_0}.')
    
    ## Making plot ##
    fig, ax = plt.subplots(figsize=(6,4), dpi=300)
    ax.plot(ln_lattice_size_array, per_run_median_equilibrated_heat_capacities_per_spin_array, '-x')

    # Plotting with the calculated intercept 
    ax.plot(x_lattice_sizes_straightline, straight_line(x_lattice_sizes_straightline, gradient_C_0, C_0_intercept), linestyle='--', color='black', label='Calculated C_0')
    ax.plot(x_lattice_sizes_straightline, straight_line(x_lattice_sizes_straightline, onsager_C_0, C_0_intercept), linestyle='--', color='red', label='Onsager C_0')
    ax.legend(loc='upper right')
    ax.grid(ls=':', color='0.9')
    ax.set_title('ln(Lattice size D) vs Heat Capacity MMC')
    ax.set_xlabel('ln(Lattice Size (side length^2)')
    ax.set_ylabel('Heat capacity per spin')

    # Saving figure
    fig.savefig(f'ln(Lattice size D) vs heat capacity per spin, filename={filename}, MMC.png', dpi=300)

    return fig, ax

# Function to run parallelised lattice side length change investigation (calculating the Onsager critical exponent from lattice size changes) using the wolff algorithm

In [21]:
def run_parallelised_lattice_side_length_change_investigation_simulations_WMC(parameter_file, **kwargs):

    """Calls the run_single_temp_investigation_simulation(args) function N cores times per instance and packages the results up into lists.
    Once enough instances have been called that the full temperature sweep has been completed, the lists of median magnetisations, 
    median magnetic susceptibilities and median heat capacities are converted into arrays and used for critical temperature and critical 
    exponent calculations.
    
    In order to run the run_single_temp_investigation_simulation(args) function in parallel, it uses all available cpu cores and the pool module 
    from multiprocessing package.
    
    Returns plots of magnetisations and magnetic susceptibility vs temperatures."""

    # Getting the plot mode - condition to show / hide the diagnostic plots
    plot_mode = kwargs.get('plot_mode', None)

    # Setting up ICS and the lattice from the file 
    initial_lattice_side_length, magnetic_field_B, temperature, lattice_side_length_step, exchange_constant_J, k_b, u_b, nsteps, nruns, filename = read_lattice_side_length_investigative_file(parameter_file)

    # Making a list to store temperatures
    run_lattice_side_lenths_list = []

    # Appending to the list using lattice side length step and the amount of runs
    for i in range(nruns):

        run_lattice_side_lenths_list.append(initial_lattice_side_length + i*lattice_side_length_step)

    # Turning the list into an array as this will be useful later
    run_lattice_side_lengths_array = np.asarray(run_lattice_side_lenths_list)

    ## Setting up parallelised simulation runs ##

    # Runs for worker function defined in the file 
    single_lattice_size_run_args = [(temperature, lattice_side_length, magnetic_field_B, exchange_constant_J, k_b, u_b, nsteps) for lattice_side_length in run_lattice_side_lengths_array]

    # Printing out initialisation message for parallelised simulations
    print(f'Parallelising {nruns} lattice size runs across {n_cores} CPU cores using the wolff algorithm.')
    print(f'Lattice size range: {(run_lattice_side_lengths_array[0])**2} --> {(run_lattice_side_lengths_array[-1])**2}.')
    print(f'Per-simulation parameters: temperature={temperature}, lattice side length step={lattice_side_length_step}, magnetic_field_B={magnetic_field_B}, exchange_constant_J={exchange_constant_J}, k_b={k_b}, u_b={u_b}, nsteps={nsteps}')

    # Running multiple simulations in parallel
    with Pool(processes=os.cpu_count()) as pool:
        
        # Calling the imported parallelisable worker function to run multiple simulations at the same time using multiprocessing
        # This completes the runs across multiple temperatures much faster than using one core only
        per_run_median_quantites = list(tqdm(pool.imap(run_WMC_lattice_side_length_investigation_worker, single_lattice_size_run_args), total=nruns, desc='Running parallelised wolff algorithm lattice size sweep:'))

    # Unpacking results from the list generated using multiprocessing
    # This has to be done in the same order as the run_single_temp_investigation_simulation(args) returns them in.
    # No need for indexing because the worker function for lattice side length only returns one quantity - The heat capacity
    per_run_median_equilibrated_heat_capacities_per_spin_array = np.array([median_results for median_results in per_run_median_quantites])

    # Computing the lattice sizes
    lattice_size_array = np.square(run_lattice_side_lengths_array)

    # Computing the per element logairthm of the lattice sizes
    ln_lattice_size_array = np.log(lattice_size_array)

    # Calculating C_0
    gradient_C_0, error_gradient_C_0, C_0_intercept, error_C_0_intercept = calculate_linear_regress_gradient_fit(ln_lattice_size_array, per_run_median_equilibrated_heat_capacities_per_spin_array)

    # Storing Onsager's 1944 value of C_0 - Half of the critical amplitude
    onsager_C_0 = 0.24725

    ## Error Calculation ##
    error_C_0 = np.abs(gradient_C_0 - onsager_C_0)

    # Generating array of x coordinates for plotting straight line
    x_lattice_sizes_straightline = np.linspace(np.min(ln_lattice_size_array), np.max(ln_lattice_size_array), num=1001)

    # Printing out the results for the calculated C_0 and its error compared to the onsager value.
    print('*** C_0 results ***')
    print(f'Calculated value of C_0 is {gradient_C_0}.')
    print(f'Onsager value of C_0 is {onsager_C_0}.')
    print(f'Error in calculated value of C_0 is {error_C_0}.')
    
    ## Making plot ##
    fig, ax = plt.subplots(figsize=(6,4), dpi=300)
    ax.plot(ln_lattice_size_array, per_run_median_equilibrated_heat_capacities_per_spin_array, '-x')

    # Plotting with the calculated intercept 
    ax.plot(x_lattice_sizes_straightline, straight_line(x_lattice_sizes_straightline, gradient_C_0, C_0_intercept), linestyle='--', color='black', label='Calculated C_0')
    ax.plot(x_lattice_sizes_straightline, straight_line(x_lattice_sizes_straightline, onsager_C_0, C_0_intercept), linestyle='--', color='red', label='Onsager C_0')
    ax.legend(loc='upper right')
    ax.grid(ls=':', color='0.9')
    ax.set_title('ln(Lattice size D) vs Heat Capacity WMC')
    ax.set_xlabel('ln(Lattice Size (side length^2)')
    ax.set_ylabel('Heat capacity per spin')

    # Saving figure
    fig.savefig(f'ln(Lattice size D) vs heat capacity per spin, filename={filename}, WMC.png', dpi=300)

    return fig, ax

# Validation run - critical temperature of analytical solution to the 2D ising model using the metropolis algorithm for a 32x32 lattice

In [ ]:
parameter_file = 'ising_sim_side_len_32_temp_investigation_MMC_ICS.txt'
run_parallelised_temp_change_investigation_simulations_MMC(parameter_file)

Parallelising 500 temperature investigation runs across 24 CPU cores using the metropolis algorithm.
Temperature range: 2.0 --> 2.499.
Per-simulation parameters: temp_step=0.001K, lattice_side_length=32, magnetic_field_B=0.0, exchange_constant_J=1.0, k_b=1.0, u_b=0.0, nsteps=15000000.


Running parallelised metropolis algorithm temperature sweep::   0%|          | 0/500 [00:00<?, ?it/s]

# Validation run - investigating the effect of changing lattice size on the value of C_0 and the heat capacity using the metropolis algorithm 

In [ ]:
parameter_file = 'ising_sim_C_0_side_length_investigation_MMC_ICS.txt'
run_parallelised_lattice_side_length_change_investigation_simulations_MMC(parameter_file)

# Validation run - critical temperature of analytical solution to the 2D ising model using the wolff algorithm

In [ ]:
parameter_file = 'ising_sim_side_len_64_temp_investigation_WMC_ICS.txt'
run_parallelised_temp_change_investigation_simulations_WMC(parameter_file)

# Validation run - investigating the effect of changing lattice size on the value of C_0 and the heat capacity using the wolff algorithm

In [ ]:
parameter_file = 'ising_sim_C_0_side_length_investigation_WMC_ICS.txt'
run_parallelised_lattice_side_length_change_investigation_simulations_WMC(parameter_file)

# Visualising how changing lattice size impacts the simulations using the metropolis algorithm, whilst remaining at the Onsager critical temperature

In [ ]:
# 16x16 lattice
parameter_file = 'ising_single_sim_side_len_16_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# 32x32 lattice
parameter_file = 'ising_single_sim_side_len_32_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# 64x64 lattice
parameter_file = 'ising_single_sim_side_len_64_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# 128x128 lattice
parameter_file = 'ising_single_sim_side_len_128_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

# Visualising the effect of changing the temperature for a standard lattice size of 64x64 spins using the metropolis algorithm 

In [ ]:
# 0.001K (close to absolute 0)
parameter_file = 'ising_single_sim_T_0001_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# 1K (below tcrit)
parameter_file = 'ising_single_sim_T_1_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# 2.269185K (tcrit)
parameter_file = 'ising_single_sim_T_crit_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# 3K (above tcrit)
parameter_file = 'ising_single_sim_T_3_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

# Visualising the effect of changing exchange interaction constant J for a standard lattice size of 64x64 spins using the metropolis algorithm, whilst remaining at the onsager critical temperature

In [ ]:
# J = 0.01
parameter_file = 'ising_single_sim_J_001_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# J = 0.1
parameter_file = 'ising_single_sim_J_01_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# J = 10
parameter_file = 'ising_single_sim_J_10_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# J = 100
parameter_file = 'ising_single_sim_J_100_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

# Visualising the effect of changing the magnetic induction field strength B for a standard lattice size of 64x64 spins using the metropolis algorithm, whilst remaining at the onsager critical temperature

In [ ]:
# B = 0.01 (comparable to a fridge magnet) 
parameter_file = 'ising_single_sim_B_001_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# B = -0.01 (comparable to a fridge magnet) 
parameter_file = 'ising_single_sim_B_-001_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# B = 1 (comparable to an MRI machine)
parameter_file = 'ising_single_sim_B_1_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

In [ ]:
# B = -1 (comparable to an MRI machine)
parameter_file = 'ising_single_sim_B_-1_MMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_MMC(parameter_file)

# Visualising how changing lattice size impacts the simulations using the wolff algorithm, whilst remaining at the onsager critical temperature

In [ ]:
# 16x16 lattice 
parameter_file = 'ising_single_sim_side_len_16_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

In [ ]:
# 32x32 lattice 
parameter_file = 'ising_single_sim_side_len_32_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

In [ ]:
# 64x64 lattice
parameter_file = 'ising_single_sim_side_len_64_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

In [ ]:
# 128x128 lattice
parameter_file = 'ising_single_sim_side_len_128_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

# Visualising the effect of changing the temperature for a standard lattice size of 64x64 spins using the wolff algorithm

In [ ]:
# 0.001K (close to absolute 0)
parameter_file = 'ising_single_sim_T_0001_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

In [ ]:
# 1K
parameter_file = 'ising_single_sim_T_1_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

In [ ]:
# 2.269185K (close to tcrit)
parameter_file = 'ising_single_sim_T_crit_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

In [ ]:
# 3K
parameter_file = 'ising_single_sim_T_3_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

# Visualising the effect of changing exchange interaction constant J for a standard lattice size of 64x64 spins using the wolff algorithm whilst remaining at the onsager critical temperature

In [ ]:
# J = 0.01 
parameter_file = 'ising_single_sim_J_001_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

In [ ]:
# J = 0.1
parameter_file = 'ising_single_sim_J_01_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

In [ ]:
# J = 10
parameter_file = 'ising_single_sim_J_10_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

In [ ]:
# J = 100
parameter_file = 'ising_single_sim_J_100_WMC_ICS.txt'
run_single_ising_model_simulation_with_lattice_visualisation_WMC(parameter_file)

# The wolff algorithm is not designed to account for magnetic fields, it is only designed to mitigate critical slowing and escape from local optima. Therefore the metropolis algorithm only is used to investigate the effects of B-fields.